# Figure 3 and Supplementary Figures S7–S9: rhythmic alternative splicing

This self-contained Python notebook regenerates the analyses and publication panels
for rhythmic splicing, splicing machinery, gene-expression coupling, and clock-TF
binding. Run all cells in the project's Docker **Python 3** kernel. No legacy
notebook, plotting script, or figure image is required at runtime.

The primary AS inputs are processed rMATS JC tables and their exact sample-order
lists. By default, harmonic regression is recomputed from these tables. The notebook
also consumes the current processed CHIRAL phase estimates, expression-regression
tables, GO/Reactome/KEGG enrichment tables, and ChIP-Atlas peak–gene matrix used by
the cohort analysis. These are explicit upstream analysis inputs, not all direct
outputs of the checked-in `process.smk`. This notebook does not rerun RNA alignment,
CHIRAL phase inference, expression normalization, or the upstream enrichment tests.
Supply the processed inputs separately when publishing this notebook.

All downstream statistics, figures, and source tables are computed here. Inputs are
hashed and checked, legacy data are preserved, and new outputs go under
`data/publication/fig03_rhythmic_alternative_splicing/` and
`data/figures/publication/fig03_rhythmic_alternative_splicing/`.

**Analysis conventions.** AS rhythmicity uses nominal *p* < 0.05; expression rhythmicity
uses BH *q* < 0.05 and the existing expression-amplitude cutoff. Splicing machinery
uses nominal *p* < 0.05. The legacy AS filters use PSI availability, at least one
read-covered observation, and across-project PSI variance/range before selecting
the phase-matched cohort. The read threshold is not a requirement for 50% of samples
to pass coverage. Parameters and filtering counts remain inspectable below.
Phase is in hours on a 24-hour circle. Axial R₂ is the magnitude of the second
circular moment, not regression R². The clock-TF bars count matched **event rows**;
hosts with multiple events therefore contribute more than once.

**Panel 3E.** The individual 3 × 1.5-module export shows real SLK coverage
and measured splice-junction counts from four indexed genomic BAMs. The assembled
figure includes the same real panel E. First-to-third exon skipping connections
are drawn above the coverage tracks; the inclusion connections remain below.
Configure the hypothetical BAM directory and four file paths in the first code
cell. BAM extraction code is included; a validated, portable source-table bundle also
allows rendering without access to the original BAMs. BAM provenance includes
header/index hashes and file metadata; full BAM hashes were not computed.

**Figure 3 layout.** Row 1: A–D, each 1 × 1 module. Row 2: E is 3 × 1.5
and F is 1 × 1.5 modules. Row 3: G–J, each 1 × 1. Row 4: K–N, each 1 × 1.
The assembled figure is 210 × 236.25 mm. The 1.5-module height is an explicit
author-requested exception to whole-square heights.

**Publication style.** All text sizes are configured once and are at least 7 pt.
Panels use 52.5 mm square modules. Every individual panel and assembled figure is
displayed inline and exported as PNG without panel letters and PDF with panel
letters, using `Figure{number}{panel}_{content_related}` filenames.

## 1. Setup and processed-input contract

In [ ]:
# Cell 1: Setup, parameters, and publication style
from pathlib import Path
import os
import sys
import json
import time
import hashlib
import platform
import importlib.metadata
import warnings
import copy
import numpy as np
import pandas as pd
import matplotlib
import matplotlib.pyplot as plt
from matplotlib.text import Text
from scipy import stats
from IPython.display import display, Image, Markdown

START_TIME = time.time()
root_candidates = [os.environ.get('CHRONO_PROJECT_ROOT', ''), str(Path.cwd()),
                   str(Path.cwd().parent),
                   str(Path(os.environ.get('WORKSPACE_ROOT', '/home/wdeng3/workspace/Codespace')) / 'PDAC_Rhythmicity')]
PROJECT_ROOT = next((Path(p).resolve() for p in root_candidates
                     if p and (Path(p) / 'notebook_publication').is_dir()), None)
if PROJECT_ROOT is None:
    raise FileNotFoundError('Set CHRONO_PROJECT_ROOT to the publication repository root.')
DATA_DIR = Path(os.environ.get('CHRONO_DATA_DIR', str(PROJECT_ROOT / 'data'))).resolve()

# Panel E BAM inputs: replace this hypothetical directory with your BAM folder,
# or set CHRONO_SLK_BAM_DIR before starting the notebook. Each BAM needs its index.
SLK_BAM_DIR = Path(os.environ.get('CHRONO_SLK_BAM_DIR', '/path/to/figure3E_bams'))
SLK_BAM_PATHS = {
    'TCGA-FB-A545-01A': SLK_BAM_DIR / 'TCGA-FB-A545-01A.bam',
    'TCGA-YB-A89D-01A': SLK_BAM_DIR / 'TCGA-YB-A89D-01A.bam',
    'TCGA-FB-A7DR-01A': SLK_BAM_DIR / 'TCGA-FB-A7DR-01A.bam',
    'TCGA-H6-A45N-01A': SLK_BAM_DIR / 'TCGA-H6-A45N-01A.bam',
}
# True permits figure-only regeneration from the validated BAM-derived tables
# when the original BAMs are unavailable. False requires the configured BAMs.
SLK_USE_SOURCE_CACHE = True
ANALYSIS_SLUG = 'fig03_rhythmic_alternative_splicing'
TABLE_DIR = PROJECT_ROOT / 'data' / 'publication' / ANALYSIS_SLUG
FIGURE_DIR = PROJECT_ROOT / 'data' / 'figures' / 'publication' / ANALYSIS_SLUG
LOG_DIR = PROJECT_ROOT / 'data' / 'logs' / 'publication' / ANALYSIS_SLUG
for directory in (TABLE_DIR, FIGURE_DIR, LOG_DIR):
    directory.mkdir(parents=True, exist_ok=True)

CONFIG = dict(seed=3303, bootstrap_n=2000, curve_bootstrap_n=300, permutation_n=10000,
              as_p=0.05, expression_q=0.05, expression_min_amplitude=0.1,
              recompute_as=os.environ.get('CHRONO_RECOMPUTE_AS', 'true').lower() == 'true',
              slk_mode='bam_coverage',
              circular_bandwidth=0.3)
COHORTS = ['CPTAC_Tumor', 'TCGA_Tumor', 'GTEx_Normal']
COHORT_INFO = {
    'CPTAC_Tumor': dict(project='CPTAC', subcohort='CPTAC-3_Tumor', label='CPTAC tumor', colour='#0072B2'),
    'TCGA_Tumor': dict(project='TCGA', subcohort='TCGA-PAAD_Tumor', label='TCGA tumor', colour='#D55E00'),
    'GTEx_Normal': dict(project='GTEx', subcohort='GTEx_Normal', label='GTEx normal', colour='#009E73'),
}
PANEL_MM = 210 / 4
FONT = dict(base=8, title=9, axis_title=8, tick=7, legend=7, annotation=7, tag=10, subtitle=7)
FONT_FAMILY = 'DejaVu Sans'
assert min(FONT.values()) >= 7
np.random.seed(CONFIG['seed'])
plt.rcParams.update({
    'font.family': FONT_FAMILY, 'font.size': FONT['base'],
    'axes.titlesize': FONT['title'], 'axes.titleweight': 'bold',
    'axes.labelsize': FONT['axis_title'], 'xtick.labelsize': FONT['tick'],
    'ytick.labelsize': FONT['tick'], 'legend.fontsize': FONT['legend'],
    'legend.title_fontsize': FONT['legend'], 'axes.linewidth': 0.5,
    'axes.spines.top': False, 'axes.spines.right': False,
    'lines.linewidth': 0.8, 'figure.dpi': 120, 'savefig.dpi': 300,
    'pdf.fonttype': 42, 'ps.fonttype': 42, 'figure.facecolor': 'white',
    'axes.facecolor': 'white', 'savefig.facecolor': 'white',
    'figure.constrained_layout.h_pad': 0.025, 'figure.constrained_layout.w_pad': 0.025,
    'figure.constrained_layout.hspace': 0.015, 'figure.constrained_layout.wspace': 0.015,
})
PANELS = {}
EXPORT_MANIFEST = []
print('Processed input directory:', DATA_DIR)
print('AS source mode:', 'recompute from rMATS JC' if CONFIG['recompute_as'] else 'processed AS fit cache')
print('Global fonts (pt):', FONT)
print('Publication module:', PANEL_MM, 'mm')


## 2. Recompute AS fits and downstream comparisons

Sample positions are validated against each rMATS vector before fitting. Gene IDs
are unversioned Ensembl gene identifiers; gene symbols are display labels. Current
expression-regression tables refer to the merged log2 CPM/TMM analysis. The
heatmaps bin PSI into numeric hours 0–23 and z-score each event across phase bins.
Missing phase bins remain distinguishable from measured values.

The overall SE/expression phase panels (3J/K) use separate marginal populations.
S9A–D restrict events to rhythmic expression hosts and use unique host genes for
the host-phase distribution. S9I–K show event-specific signed phase differences
for all included AS event types. The amplitude comparison uses
`log2(2 × expression Amplitude + 0.01)`, then normalizes within cohort for the
accumulation curves; bootstrap effect sizes compare the two within-cohort groups.

In [ ]:
# Cell 2: AS and expression analysis
# Self-contained analysis functions.  The surrounding notebook supplies DATA_DIR,
# TABLE_DIR, CONFIG, COHORTS, COHORT_INFO, np, pd, stats, and plt.
from pathlib import Path
from collections import OrderedDict
import re

EVENT_COLUMNS = {
    'SE': ('SE.MATS.JC.txt', ['upstream_to_target_count', 'target_to_downstream_count', 'upstream_to_downstream_count'], 'exonStart_0base', 'exonEnd'),
    'A3SS': ('A3SS.MATS.JC.txt', ['across_short_boundary_count', 'long_to_flanking_count', 'short_to_flanking_count'], 'longExonStart_0base', 'longExonEnd'),
    'A5SS': ('A5SS.MATS.JC.txt', ['across_short_boundary_count', 'long_to_flanking_count', 'short_to_flanking_count'], 'longExonStart_0base', 'longExonEnd'),
    'MXE': ('MXE.MATS.JC.txt', ['upstream_to_first_count', 'first_to_downstream_count', 'upstream_to_second_count', 'second_to_downstream_count'], '1stExonStart_0base', '1stExonEnd'),
    'RI': ('RI.MATS.JC.txt', ['upstream_to_intron_count', 'intron_to_downstream_count', 'upstream_to_downstream_count'], 'riExonStart_0base', 'riExonEnd'),
}
FILTERS = {'SE': (30, .5, .05, .1), 'A3SS': (30, .5, .01, .1),
           'A5SS': (30, .5, .01, .1), 'MXE': (30, .3, .01, .1), 'RI': (50, .5, .005, .1)}
MACHINERY_CATEGORIES = OrderedDict({
    'Core Spliceosome': 'SNRPA SNRPB SNRPC SNRPD1 SNRPD2 SNRPD3 SNRPE SNRPF SNRPG SNRPA1 SNRPB2 SF3A1 SF3A2 SF3A3 SF3B1 SF3B2 SF3B3 SF3B4 SF3B5 SF3B6 PRPF3 PRPF4 PRPF6 PRPF8 PRPF31 SNRNP200 EFTUD2 DDX23 PRPF19 CDC5L PLRG1 BCAS2 CWC15 SYF2'.split(),
    'SR Proteins': 'SRSF1 SRSF2 SRSF3 SRSF4 SRSF5 SRSF6 SRSF7 SRSF8 SRSF9 SRSF10 SRSF11 SRSF12 TRA2A TRA2B'.split(),
    'hnRNP Proteins': 'HNRNPA1 HNRNPA2B1 HNRNPC HNRNPD HNRNPF HNRNPH1 HNRNPH2 HNRNPK HNRNPL HNRNPM HNRNPU PTBP1 PTBP2'.split(),
    'Other Splicing Regulators': 'RBFOX1 RBFOX2 MBNL1 MBNL2 NOVA1 NOVA2 QKI CELF1 CELF2 ESRP1 ESRP2 RBM4 RBM5 RBM10 RBM25 PRPF40A U2AF1 U2AF2 SRRM1 SRRM2'.split(),
})

def _base_id(x):
    return re.sub(r'\..*$', '', str(x).replace('"', '').strip())

def _numvec(value):
    if value is None or pd.isna(value) or str(value) == '': return np.array([], float)
    return np.fromiter((np.nan if x in ('NA', '', 'nan') else float(x) for x in str(value).split(',')), float)

def _bh(p):
    p = np.asarray(p, float); out = np.full(p.size, np.nan); good = np.isfinite(p)
    order = np.argsort(p[good]); ranked = p[good][order]; n = len(ranked)
    q = np.minimum.accumulate((ranked * n / np.arange(1, n + 1))[::-1])[::-1]
    tmp = np.empty(n); tmp[order] = np.minimum(q, 1); out[good] = tmp
    return out

def _hour(phi): return (np.asarray(phi, float) % (2 * np.pi)) * 12 / np.pi
def _circular_difference(a, b): return ((np.asarray(a) - np.asarray(b) + 12) % 24) - 12

def _fit_harmonic(y, phase):
    keep = np.isfinite(y) & np.isfinite(phase)
    y, phase = np.asarray(y)[keep], np.asarray(phase)[keep]
    if len(y) < 10: return None
    x = np.c_[np.ones(len(y)), np.cos(phase), np.sin(phase)]
    beta, _, _, _ = np.linalg.lstsq(x, y, rcond=None); pred = x @ beta
    rss = ((y-pred)**2).sum(); reduced = ((y-y.mean())**2).sum(); df = len(y)-3
    f = ((reduced-rss)/2)/(rss/df) if rss > 0 and df > 0 else 0.
    phase_rad = np.arctan2(beta[2], beta[1]) % (2*np.pi)
    return dict(n_samples=len(y), beta0=beta[0], beta1=beta[1], beta2=beta[2],
                amplitude=float(np.hypot(beta[1], beta[2])), acrophase_rad=phase_rad,
                acrophase_hours=float(_hour(phase_rad)), p=float(stats.f.sf(max(f, 0), 2, df)) if f > 0 else 1.,
                r_squared=float(1-rss/reduced) if reduced > 0 else 0., mean=float(y.mean()), sd=float(y.std()))

def _sample_ids(project):
    f = DATA_DIR / 'AS' / project / 'all_samples.txt'
    paths = f.read_text().strip().split(',')
    ans = []
    for path in paths:
        name = Path(path.strip()).name.replace('.bam', '')
        # Current GTEx rMATS lists retain the STAR output suffix; CHIRAL/meta
        # use the preceding GTEx specimen identifier.
        if '.Aligned.sortedByCoord.out' in name: name = name.split('.Aligned.sortedByCoord.out', 1)[0]
        ans.append(Path(path.strip()).parent.name if name.startswith('Aligned') else name)
    if len(ans) != len(set(ans)): raise ValueError(f'{project}: canonical rMATS sample IDs collide after filename normalization')
    return ans

def _phase_metadata():
    meta = pd.read_csv(DATA_DIR / 'Merged' / 'GTEx_TCGA_GEO_merged_meta.csv')
    sample_col = 'Sample' if 'Sample' in meta else meta.columns[-1]
    phi = pd.read_csv(DATA_DIR / 'Merged' / 'CHIRAL_dual_penalty_phi_12h_geo.csv')
    d = {}; phase_ids=[]
    for row in phi.itertuples(index=False):
        sample, value = str(getattr(row, 'sample')).strip('"'), float(getattr(row, 'phi'))
        for each in sample.split(','):
            each=each.strip(); phase_ids.append(each); d[each] = value
    out = meta[[sample_col, 'Info']].rename(columns={sample_col: 'sample_id', 'Info': 'subcohort'}).copy()
    out['sample_id'] = out.sample_id.astype(str).str.strip('"'); out['phase_rad'] = out.sample_id.map(d); out['phase_h'] = _hour(out.phase_rad)
    if out.sample_id.duplicated().any() or len(phase_ids) != len(set(phase_ids)): raise ValueError('metadata or CHIRAL phase table contains duplicate sample IDs')
    return out

def _cohort_samples(project, subcohort, phase_meta):
    ids = _sample_ids(project); md = phase_meta.query('subcohort == @subcohort').set_index('sample_id')
    if len(ids) != len(set(ids)): raise ValueError(f'{project}: all_samples.txt contains duplicate IDs')
    positions = np.array([i for i, s in enumerate(ids) if s in md.index and np.isfinite(md.at[s, 'phase_rad'])], int)
    samples = pd.DataFrame({'sample_id': [ids[i] for i in positions], 'phase_rad': [md.at[ids[i], 'phase_rad'] for i in positions]})
    samples['phase_h'] = _hour(samples.phase_rad)
    if len(positions) < 10: raise ValueError(f'{subcohort}: only {len(positions)} rMATS samples have CHIRAL phases')
    return ids, positions, samples

def _event_rows_from_jc(path, event_type, positions, phases, full_n, selected_id=None):
    min_reads, prevalence, min_variance, min_range = FILTERS[event_type]; count_cols = EVENT_COLUMNS[event_type][1]
    records, raw_selected, n_raw, n_cov, n_var = [], None, 0, 0, 0
    for chunk in pd.read_csv(path, sep='\t', chunksize=5000):
        n_raw += len(chunk)
        for _, row in chunk.iterrows():
            psi = _numvec(row.get('IncLevel1'))
            counts = [_numvec(row.get(c)) for c in count_cols if c in row.index]
            if not len(psi) or not counts: continue
            if len(psi) != full_n or any(len(v) != full_n for v in counts):
                raise ValueError(f'{path.name}: PSI/count vectors must equal all_samples.txt ({full_n}); event {row.get("ID")} is malformed')
            coverage = np.sum(counts, axis=0)
            if np.sum(np.isfinite(psi)) < int(full_n * prevalence) or not np.any(np.isfinite(psi) & (coverage > min_reads)): continue
            n_cov += 1; valid = psi[np.isfinite(psi)]
            if len(valid) < 10 or np.var(valid) < min_variance or np.ptp(valid) < min_range: continue
            n_var += 1
            event_id = str(row.get('ID'))
            if event_id == str(selected_id): raw_selected = (row.copy(), psi, counts)
            fit = _fit_harmonic(psi[positions], phases)
            if fit is None: continue
            fit.update(event_id=event_id, gene_id=_base_id(row.get('GeneID', '')), gene_symbol=str(row.get('geneSymbol', '')).strip('"'), event_type=event_type,
                       chr=row.get('chr', np.nan), strand=row.get('strand', np.nan), exon_start=row.get(EVENT_COLUMNS[event_type][2], np.nan), exon_end=row.get(EVENT_COLUMNS[event_type][3], np.nan))
            fit['acrophase'] = fit['acrophase_hours']
            records.append(fit)
    df = pd.DataFrame(records)
    if len(df): df['q'] = _bh(df.p); df = df.sort_values('q').reset_index(drop=True)
    return df, raw_selected, dict(raw=n_raw, coverage=n_cov, variable=n_var, fitted=len(df))

def _cache_events(path, event_type):
    d = pd.read_csv(path).rename(columns={'p_value':'p', 'qvalue':'q'})
    required={'event_id','gene_id','gene_symbol','p','q','amplitude'}
    if not required.issubset(d.columns): raise ValueError(f'Invalid AS development cache {path}: missing {sorted(required-set(d.columns))}')
    d['event_id'] = d.event_id.astype(str); d['gene_id'] = d.gene_id.map(_base_id); d['event_type'] = event_type
    if 'acrophase_hours' not in d: d['acrophase_hours'] = _hour(d.acrophase)
    d['acrophase_rad'] = pd.to_numeric(d['acrophase'])
    d['acrophase'] = d['acrophase_hours'] % 24
    return d

def _read_se_vectors(path, positions, full_n, wanted, selected_id=None):
    """Read PSI (and selected-event JC counts) with the validated rMATS order."""
    vectors, selected = {}, None
    usecols=['ID','IncLevel1'] + EVENT_COLUMNS['SE'][1]
    for chunk in pd.read_csv(path, sep='\t', usecols=usecols, chunksize=5000):
        chunk=chunk.loc[chunk['ID'].astype(str).isin(set(wanted)|{str(selected_id)})]
        for _, row in chunk.iterrows():
            eid=str(row['ID'])
            if eid not in wanted and eid != str(selected_id): continue
            psi=_numvec(row['IncLevel1']); counts=[_numvec(row[c]) for c in EVENT_COLUMNS['SE'][1]]
            if len(psi)!=full_n or any(len(v)!=full_n for v in counts): raise ValueError(f'{path.name}: malformed SE vector for {eid}')
            if eid in wanted: vectors[eid]=psi[positions]
            if eid == str(selected_id): selected=(row.copy(),psi,counts)
    return vectors, selected

def _expression(cohort, info):
    sub = info['subcohort']; f = DATA_DIR/'Merged'/'regression'/f'all_genes_regression_periodic_12h_{sub}.txt'
    d = pd.read_csv(f, sep='\t').rename(columns={'Gene':'gene_symbol', 'ENSG':'gene_id', 'Amplitude':'amplitude'})
    d['gene_id'] = d.gene_id.map(_base_id); d['acrophase'] = d.acrophase % 24
    d['p'] = pd.to_numeric(d.p); d['q'] = pd.to_numeric(d.q); d['cohort'] = cohort
    return d

def _legacy_sig_expression(cohort, info, all_expression=None):
    f=DATA_DIR/'Merged'/'regression'/f"sig_genes_regression_periodic_12h_{info['subcohort']}.txt"
    d=pd.read_csv(f,sep='\t').rename(columns={'Gene':'gene_symbol','ENSG':'gene_id','Amplitude':'amplitude'})
    d['gene_id']=d.gene_id.map(_base_id); d['acrophase']=pd.to_numeric(d.acrophase)%24; d['p']=pd.to_numeric(d.p); d['q']=pd.to_numeric(d.q); d['cohort']=cohort
    if d.gene_id.duplicated().any(): raise ValueError(f'{f}: duplicate ENSG identifiers')
    all_expression = _expression(cohort, info) if all_expression is None else all_expression
    selected = all_expression.loc[(all_expression.q < CONFIG['expression_q']) &
                                  (all_expression.amplitude > CONFIG['expression_min_amplitude'])].copy()
    if selected.gene_id.duplicated().any():
        raise ValueError(f'{cohort}: duplicate ENSG identifiers in rhythmic expression')
    if CONFIG['expression_q'] == .05 and CONFIG['expression_min_amplitude'] == .1:
        if set(selected.gene_id) != set(d.gene_id):
            raise ValueError(f'{cohort}: default expression calls differ from the declared significant-gene input')
    return selected

def _resample_axis(values, n, rng):
    values = np.asarray(values, float); values = values[np.isfinite(values)]
    if len(values) < 3: return np.nan, np.nan, np.nan
    def r2(v):
        z = np.exp(4j*np.pi*np.asarray(v)/24).mean(); return float(abs(z))
    obs = r2(values)
    if n <= 0: return obs, np.nan, np.nan
    boots = np.array([r2(rng.choice(values, len(values), True)) for _ in range(n)])
    return obs, *np.quantile(boots, [.025, .975])

def _watson_wheeler(x, y):
    # Rank-based two-sample Watson--Wheeler approximation, compatible with legacy interpretation.
    x, y = np.asarray(x, float), np.asarray(y, float); x=x[np.isfinite(x)]; y=y[np.isfinite(y)]
    if min(len(x), len(y)) < 2: return np.nan, np.nan
    angles = np.r_[x, y] * 2*np.pi/24; labels = np.r_[np.zeros(len(x)), np.ones(len(y))]
    ranks = stats.rankdata(angles); n1, n2, n = len(x), len(y), len(angles); ranked=2*np.pi*ranks/n
    score=0.
    for label, n_group in ((0,n1),(1,n2)):
        c=np.cos(ranked[labels==label]).sum(); s=np.sin(ranked[labels==label]).sum(); score+=(c*c+s*s)/n_group
    w=2*score; return float(w), float(stats.chi2.sf(w, 2))

def _axis_hour(values):
    values=np.asarray(values,float); values=values[np.isfinite(values)]
    if not len(values): return np.nan
    return float((np.angle(np.exp(4j*np.pi*values/24).mean())/2)%(np.pi)*12/np.pi)

def _heatmap(events, raw_se, samples):
    bins = np.arange(24); matrix=[]; ids=[]
    phase_h = samples.phase_h.to_numpy(); order = np.argsort(phase_h)
    # only SE rows can receive raw PSI here; re-read compactly in caller is avoided by selected heatmap cache
    phase_order = events.sort_values(['acrophase_hours', 'event_id'])['event_id'].astype(str).tolist()
    for eid in phase_order:
        if eid not in raw_se: continue
        psi = raw_se[eid]
        vals=np.asarray(psi,float); means=[]
        hour_bins=np.floor(phase_h).astype(int)%24
        for b in bins:
            mask=(hour_bins == b) & np.isfinite(vals)
            means.append(vals[mask].mean() if mask.any() else np.nan)
        means=np.asarray(means); matrix.append((means-np.nanmean(means))/(np.nanstd(means)+1e-8)); ids.append(str(eid))
    matrix=np.asarray(matrix); return dict(matrix=matrix, event_ids=np.asarray(ids), phase_hours=bins)

def build_analysis():
    """Load/recompute Figure 3 analysis data; creates no files or figures."""
    rng=np.random.default_rng(CONFIG['seed']); phase_meta=_phase_metadata(); events={}; expression={}; rhythmic_expression={}; qc=[]; sample_store={}; raw_slk=None; slk_context=None
    use_recompute=bool(CONFIG.get('recompute_as', True)); heatmap_raw={}
    for cohort in COHORTS:
        info=COHORT_INFO[cohort]; project=info['project']; ids, positions, samples=_cohort_samples(project, info['subcohort'], phase_meta); phases=samples.phase_rad.to_numpy(); sample_store[cohort]=(positions,samples)
        parts=[]
        for etype,(filename,*_) in EVENT_COLUMNS.items():
            jc=DATA_DIR/'AS'/project/'post'/filename; cache=DATA_DIR/'AS'/project/'rhythmic_results'/cohort/etype/'all_results.csv'
            if use_recompute:
                fit, raw, counts=_event_rows_from_jc(jc, etype, positions, phases, len(ids), selected_id='150506' if (cohort=='TCGA_Tumor' and etype=='SE') else None)
                mode='recomputed_JC'
            else:
                fit=_cache_events(cache, etype); counts={'raw':np.nan,'coverage':np.nan,'variable':np.nan,'fitted':len(fit)}; mode='validated_cache'
            if etype=='SE':
                wanted=set(fit.loc[fit.p < CONFIG['as_p'], 'event_id'].astype(str))
                heatmap_raw[cohort], selected=_read_se_vectors(jc, positions, len(ids), wanted, '150506' if cohort=='TCGA_Tumor' else None)
                if cohort=='TCGA_Tumor' and selected is not None: raw_slk=selected; slk_context=(positions,samples,fit)
            parts.append(fit); qc.append({'cohort':cohort,'event_type':etype,'source_mode':mode,**counts,'raw_p_lt_0_05':int((fit.p<CONFIG['as_p']).sum())})
        events[cohort]=pd.concat(parts,ignore_index=True) if parts else pd.DataFrame(); expression[cohort]=_expression(cohort,info)
        rhythmic_expression[cohort]=_legacy_sig_expression(cohort,info,expression[cohort])
        print(f"{cohort}: AS events={len(events[cohort]):,}, nominal rhythmic={int((events[cohort].p < CONFIG['as_p']).sum()):,}; expression rhythmic={len(rhythmic_expression[cohort]):,}")
    machinery=[]
    lookup={g:cat for cat,genes in MACHINERY_CATEGORIES.items() for g in genes}
    for cohort,d in expression.items():
        z=d[d.gene_symbol.isin(lookup)].copy(); z['category']=z.gene_symbol.map(lookup); z['cohort']=cohort; machinery.append(z[['cohort','category','gene_symbol','p','amplitude','acrophase']])
    machinery=pd.concat(machinery,ignore_index=True)
    peak=DATA_DIR/'enrichment'/'chip_atlas_enrichment'/'peak_gene_matrix.txt'
    if not peak.exists(): raise FileNotFoundError(f'Required external ChIP-Atlas peak matrix is absent: {peak}')
    pm=pd.read_csv(peak,sep='\t',index_col=0); peakset=set(map(_base_id,pm.index[pm.sum(axis=1)>0]))
    matched={}; metrics=[]; amp_rows=[]; effect_rows=[]; effect_boots={}; effect_estimates={}; binding=[]; density={}
    for cohort in COHORTS:
        ase=events[cohort].loc[events[cohort]['p'] < CONFIG['as_p']].copy(); ex=rhythmic_expression[cohort][['gene_id','acrophase','amplitude']].rename(columns={'acrophase':'expr_acrophase','amplitude':'expr_amplitude'})
        m=ase.merge(ex,on='gene_id',how='inner'); m['phase_diff']=_circular_difference(m.acrophase_hours,m.expr_acrophase); m['has_clock_peak']=m.gene_id.isin(peakset); matched[cohort]=m
        se=m[m.event_type=='SE'].copy(); se_genes=se.drop_duplicates('gene_id').copy()
        ar,al,ah=_resample_axis(se.acrophase_hours,CONFIG['bootstrap_n'],rng); gr,gl,gh=_resample_axis(se_genes.expr_acrophase,CONFIG['bootstrap_n'],rng)
        diff=ar-gr if np.isfinite(ar*gr) else np.nan; null=[]
        if min(len(se),len(se_genes))>=2:
            pooled=np.r_[se.acrophase_hours.to_numpy(), se_genes.expr_acrophase.to_numpy()]
            n_as=len(se)
            for _ in range(CONFIG['permutation_n']):
                shuffled=rng.permutation(pooled)
                null.append(_resample_axis(shuffled[:n_as],0,rng)[0]-_resample_axis(shuffled[n_as:],0,rng)[0])
        w,wp=_watson_wheeler(se.acrophase_hours,se_genes.expr_acrophase); as_axis,gene_axis=_axis_hour(se.acrophase_hours),_axis_hour(se_genes.expr_acrophase); axis_sep=abs(((as_axis-gene_axis+6)%12)-6) if np.isfinite(as_axis*gene_axis) else np.nan
        metrics.append({'cohort':cohort,'AS_R2':ar,'gene_R2':gr,'AS_ci_low':al,'AS_ci_high':ah,'gene_ci_low':gl,'gene_ci_high':gh,'delta_R2':diff,'permutation_p':((1+np.sum(np.abs(null)>=abs(diff)))/(1+len(null)) if null else np.nan),'axis_separation_h':axis_sep,'watson_p':wp,'watson_w':w})
        rhose=set(events[cohort].loc[(events[cohort].event_type=='SE') & (events[cohort].p < CONFIG['as_p']),'gene_id'])
        g=rhythmic_expression[cohort].copy(); g['has_rhythmic_se']=g.gene_id.isin(rhose); g['amplitude']=2*g.amplitude+0.01; g['log_amplitude']=np.log2(g.amplitude); g['normalized_log_amplitude']=np.log2(g.amplitude/np.median(g.amplitude)); amp_rows.append(g[['cohort','gene_id','amplitude','log_amplitude','normalized_log_amplitude','has_rhythmic_se']])
        yes=g.loc[g.has_rhythmic_se,'log_amplitude'].dropna().to_numpy(); no=g.loc[~g.has_rhythmic_se,'log_amplitude'].dropna().to_numpy(); effect=float(np.median(yes)-np.median(no)) if len(yes) and len(no) else np.nan
        boot=np.array([np.median(rng.choice(yes,len(yes),True))-np.median(rng.choice(no,len(no),True)) for _ in range(CONFIG['bootstrap_n'])]) if len(yes) and len(no) else np.array([np.nan])
        effect_rows.append({'cohort':cohort,'effect':'with_rhythmic_SE_minus_no_rhythmic_SE','ci_low':np.nanquantile(boot,.025),'ci_high':np.nanquantile(boot,.975),'estimate':effect,'n_with_rhythmic_SE':len(yes),'n_no_rhythmic_SE':len(no),'bootstrap_n':CONFIG['bootstrap_n']})
        effect_boots[cohort]=boot; effect_estimates[cohort]=effect
        for scope,frame in [('all',m),('SE',se)]:
            k=int(frame.has_clock_peak.sum()); n=len(frame); lo,hi=_wilson(k,n); binding.append({'cohort':cohort,'scope':scope,'n_with':k,'n_total':n,'frac':k/n if n else np.nan,'ci_low':lo,'ci_high':hi})
        density[cohort]={'as_hours':ase.query("event_type=='SE'").acrophase_hours.to_numpy(),'expression_hours':rhythmic_expression[cohort].acrophase.to_numpy(),'matched_se_as_hours':se.acrophase_hours.to_numpy(),'matched_se_expression_hours':se_genes.expr_acrophase.to_numpy()}
    for target in ('TCGA_Tumor','CPTAC_Tumor'):
        reference='GTEx_Normal'
        if target in effect_boots and reference in effect_boots:
            contrast=effect_boots[target]-effect_boots[reference]
            effect_rows.append({'cohort':target,'effect':f'{target}_effect_minus_{reference}_effect','ci_low':np.nanquantile(contrast,.025),'ci_high':np.nanquantile(contrast,.975),'estimate':effect_estimates[target]-effect_estimates[reference],'n_with_rhythmic_SE':np.nan,'n_no_rhythmic_SE':np.nan,'bootstrap_n':CONFIG['bootstrap_n']})
    heatmaps={c:_heatmap(events[c].loc[(events[c]['event_type']=='SE') & (events[c]['p'] < CONFIG['as_p'])],heatmap_raw.get(c,{}),sample_store[c][1]) for c in COHORTS}
    # SLK uses the original rMATS positional vector, restricted to the phase-aligned TCGA slots.
    slk={'samples':pd.DataFrame(),'fit_phase':np.array([]),'fit_psi':np.array([]),'representatives':pd.DataFrame(),'event':pd.Series(dtype=object),'junction_counts':pd.DataFrame()}
    if raw_slk is not None and slk_context is not None:
        positions,samples,fit=slk_context; row,psi,counts=raw_slk; ev=fit[fit.event_id=='150506'].iloc[0]; s=samples.copy(); s['psi']=psi[positions]; s['is_representative']=False; s['target_h']=np.nan; targets=[(ev.acrophase_hours+k)%24 for k in (0,6,12,18)]; reps=[]; used=set()
        pred=ev.beta0+ev.beta1*np.cos(s.phase_rad)+ev.beta2*np.sin(s.phase_rad); s['abs_residual']=abs(s.psi-pred)
        for target in targets:
            cand=s.loc[~s.index.isin(used) & (abs(_circular_difference(s.phase_h,target))<=2)].sort_values('abs_residual')
            if len(cand): idx=cand.index[0]; used.add(idx); s.loc[idx,['is_representative','target_h']]=[True,target]; reps.append(s.loc[idx])
        junction=pd.DataFrame({'sample_id':s.sample_id,'phase_h':s.phase_h,'target_h':s.target_h,'is_representative':s.is_representative,'UTC':counts[0][positions] if len(counts)>0 else np.nan,'TDC':counts[1][positions] if len(counts)>1 else np.nan,'UDC':counts[2][positions] if len(counts)>2 else np.nan})
        grid=np.linspace(0,24,240); slk={'samples':s[['sample_id','phase_h','psi','is_representative','target_h']],'fit_phase':grid,'fit_psi':ev.beta0+ev.beta1*np.cos(2*np.pi*grid/24)+ev.beta2*np.sin(2*np.pi*grid/24),'representatives':pd.DataFrame(reps),'event':ev,'junction_counts':junction}
    return {'events':events,'expression':expression,'rhythmic_expression':rhythmic_expression,'machinery':machinery,'machinery_categories':dict(MACHINERY_CATEGORIES),'matched':matched,'heatmaps':heatmaps,'slk':slk,'selected_event':slk['event'],'circular_metrics':pd.DataFrame(metrics),'amplitude_source':pd.concat(amp_rows,ignore_index=True),'amplitude_effects':pd.DataFrame(effect_rows),'bindings':pd.DataFrame(binding),'phase_density':density,'qc':pd.DataFrame(qc)}

def _wilson(k,n,z=1.96):
    if not n:return (np.nan,np.nan)
    p=k/n; den=1+z*z/n; mid=(p+z*z/(2*n))/den; half=z*np.sqrt(p*(1-p)/n+z*z/(4*n*n))/den
    return max(0,mid-half),min(1,mid+half)


In [ ]:
# Cell 3: AS and expression analysis
DATA = build_analysis()


In [ ]:
# Cell 4: QC, source tables, and input provenance
display(DATA['qc'])
for key in ['machinery', 'circular_metrics', 'amplitude_source', 'amplitude_effects', 'bindings', 'qc']:
    DATA[key].to_csv(TABLE_DIR / f'{key}.tsv', sep='\t', index=False)
for cohort in COHORTS:
    for key in ['events', 'expression', 'rhythmic_expression', 'matched']:
        DATA[key][cohort].to_csv(TABLE_DIR / f'{cohort}_{key}.tsv', sep='\t', index=False)
    heat = DATA['heatmaps'][cohort]
    pd.DataFrame(heat['matrix'], index=heat['event_ids'], columns=heat['phase_hours']).rename_axis('event_id').to_csv(
        TABLE_DIR / f'{cohort}_SE_phase_binned_PSI_zscore.tsv', sep='\t')
for key in ['samples', 'representatives', 'junction_counts']:
    DATA['slk'][key].to_csv(TABLE_DIR / f'SLK_{key}.tsv', sep='\t', index=False)
pd.DataFrame({'phase_h': DATA['slk']['fit_phase'], 'fitted_PSI': DATA['slk']['fit_psi']}).to_csv(
    TABLE_DIR / 'SLK_harmonic_curve.tsv', sep='\t', index=False)
clock_genes = ['ARNTL','CLOCK','NPAS2','PER1','PER2','PER3','CRY1','CRY2','NR1D1','NR1D2','DBP','TEF']
clock_control = pd.concat([DATA['expression'][c].assign(cohort=c).query('gene_symbol in @clock_genes')
                           for c in COHORTS], ignore_index=True)
clock_control.to_csv(TABLE_DIR / 'core_clock_gene_controls.tsv', sep='\t', index=False)
clock_control_display = clock_control.assign(gene_symbol=clock_control['gene_symbol'].replace({'ARNTL': 'BMAL1'}))
display(clock_control_display[['cohort','gene_symbol','acrophase','amplitude','p','q']])
display(DATA['bindings'])
display(DATA['circular_metrics'])
phase_metadata = _phase_metadata()
sample_qc_rows = []
sample_order_rows = []
for cohort in COHORTS:
    info = COHORT_INFO[cohort]
    all_ids, positions, selected = _cohort_samples(info['project'], info['subcohort'], phase_metadata)
    all_bams = (DATA_DIR/'AS'/info['project']/'all_samples.txt').read_text().strip().split(',')
    phase_map = selected.set_index('sample_id')['phase_h']
    sample_qc_rows.append(dict(cohort=cohort, rmats_samples=len(all_ids), matched_phase_samples=len(selected),
                               min_phase_h=selected.phase_h.min(), max_phase_h=selected.phase_h.max()))
    for slot, (sample, bam) in enumerate(zip(all_ids, all_bams)):
        sample_order_rows.append(dict(cohort=cohort, rmats_slot=slot, input_bam_name=Path(bam.strip()).name,
                                      sample_id=sample, selected_for_cohort=slot in set(positions),
                                      phase_h=phase_map.get(sample, np.nan)))
pd.DataFrame(sample_order_rows).to_csv(TABLE_DIR/'rmats_sample_order_manifest.tsv', sep='\t', index=False)
sample_qc = pd.DataFrame(sample_qc_rows)
sample_qc.to_csv(TABLE_DIR/'sample_overlap_qc.tsv', sep='\t', index=False)
display(sample_qc)
pd.DataFrame([dict(event_type=k, min_reads_gt=v[0], min_PSI_availability_fraction=v[1],
                   minimum_PSI_variance=v[2], minimum_PSI_range=v[3]) for k,v in FILTERS.items()]).to_csv(
    TABLE_DIR/'AS_filter_parameters.tsv', sep='\t', index=False)


In [ ]:
# Cell 5: QC, source tables, and input provenance
input_files = [DATA_DIR / 'Merged' / 'GTEx_TCGA_GEO_merged_meta.csv',
               DATA_DIR / 'Merged' / 'CHIRAL_dual_penalty_phi_12h_geo.csv',
               DATA_DIR / 'enrichment' / 'chip_atlas_enrichment' / 'peak_gene_matrix.txt']
for cohort in COHORTS:
    info = COHORT_INFO[cohort]
    input_files.append(DATA_DIR / 'Merged' / 'regression' / f'all_genes_regression_periodic_12h_{info["subcohort"]}.txt')
    input_files.append(DATA_DIR / 'Merged' / 'regression' / f'sig_genes_regression_periodic_12h_{info["subcohort"]}.txt')
    input_files.append(DATA_DIR / 'AS' / info['project'] / 'all_samples.txt')
    for event_type in ['SE', 'A3SS', 'A5SS', 'MXE', 'RI']:
        input_files.append(DATA_DIR / 'AS' / info['project'] / 'post' / f'{event_type}.MATS.JC.txt')
        if not CONFIG['recompute_as']:
            input_files.append(DATA_DIR / 'AS' / info['project'] / 'rhythmic_results' / cohort / event_type / 'all_results.csv')
input_files += [DATA_DIR / 'enrichment' / filename for filename in [
    'full_term_GO_Regression_CHIRAL_12h_geo.txt',
    'full_term_Reactome_Regression_CHIRAL_12h_plus_geo.txt',
    'full_term_KEGG_Regression_CHIRAL_12h_plus_geo.txt']]
input_manifest_rows = []
for file in sorted(set(input_files)):
    if not file.is_file():
        raise FileNotFoundError(f'Required processed input is absent: {file}')
    digest = hashlib.sha256()
    with file.open('rb') as stream:
        for block in iter(lambda: stream.read(8 * 1024 * 1024), b''):
            digest.update(block)
    input_manifest_rows.append(dict(path=str(file.relative_to(DATA_DIR)), bytes=file.stat().st_size,
                                    sha256=digest.hexdigest()))
pd.DataFrame(input_manifest_rows).to_csv(TABLE_DIR / 'processed_input_manifest.tsv', sep='\t', index=False)
print(f'Validated and hashed {len(input_manifest_rows)} processed inputs.')


## 3. Pathway enrichment post-processing

GO domain counts use nominal p ≤ 0.05. S7A selects tumor pathways at p ≤ 0.01,
with up to four terms per domain/library/tumor cohort and corresponding GTEx
points. The unsupervised text clusters are computed before the regex domain
overlay. Enrichment ratios and transformed p values are taken from the declared
processed enrichment inputs; the original enrichment tests are not rerun here.

In [ ]:
# Cell 6: Pathway sources and drawing functions
"""Pathway panels for the Figure 3 publication notebook.

This module is executed by the notebook after it defines DATA_DIR, TABLE_DIR,
FONT, CONFIG, PANELS, numpy/pandas, and matplotlib.pyplot.  It deliberately
uses the three already processed enrichment result tables described in the
Figure 3 provenance audit.  Those tables are upstream enrichment outputs, not
direct products declared by process.smk.
"""
from pathlib import Path
import re
import textwrap
import warnings


PATHWAY_COHORTS = ("GTEx_Normal", "CPTAC-3_Tumor", "TCGA-PAAD_Tumor")
PATHWAY_DOMAINS = {
    "splicing": ("Splicing", r"\bsplic\w*|spliceosom"),
    "apa_polyadenylation": ("APA / polyadenylation", r"polyadenyl|poly\(a\)|cleavage.*polyadenyl|mrna.*3'?-?end|3'?-?end.*processing"),
    "mirna": ("miRNA", r"\bmirna\b|micro[- ]?rna|rna interference|gene silencing by mirna"),
    "transcription": ("Transcription", r"transcription|rna polymerase|dna-templated|preinitiation|promoter"),
    "dna_methylation_related": ("DNA/chromatin methylation", r"\bdna methylation\b|\bdna demethylation\b|dna methyltransferase|\bdnmt\b|methylation-dependent chromatin|chromatin silencing"),
}
PATHWAY_DOMAIN_ORDER = [label for label, _ in PATHWAY_DOMAINS.values()]
FIG3A_DOMAIN_ORDER = PATHWAY_DOMAIN_ORDER[:4]
PATHWAY_SOURCES = {
    "GO": ("full_term_GO_Regression_CHIRAL_12h_geo.txt",),
    "Reactome": ("full_term_Reactome_Regression_CHIRAL_12h_plus_geo.txt", "full_term_Reactome_Regression_CHIRAL_12h_geo.txt"),
    "KEGG": ("full_term_KEGG_Regression_CHIRAL_12h_plus_geo.txt", "full_term_KEGG_Regression_CHIRAL_12h_geo.txt"),
}
_EXTRA_STOPWORDS = {
    "regulation", "positive", "negative", "process", "pathway", "pathways", "signaling", "response", "cell", "cells", "cellular", "human", "involved", "via", "mediated", "dependent", "activity", "protein", "proteins", "complex", "complexes", "system",
}


def _clean_group_name(value):
    value = re.sub(r"^sig_genes_regression_periodic_", "", str(value))
    value = re.sub(r"\.txt$", "", value)
    return re.sub(r"^12h_", "", value)


def _load_one_pathway_source(source, candidates, enrichment_dir):
    """Select the same highest-coverage candidate used by legacy cell 47."""
    choices = []
    for filename in candidates:
        path = enrichment_dir / filename
        if not path.exists():
            continue
        raw = pd.read_csv(path, sep="\t")
        needed = {"Group", "ID", "GO_Term", "Term", "E_ratio", "P_value"}
        absent = needed.difference(raw.columns)
        if absent:
            raise ValueError(f"{path} lacks expected enrichment columns: {sorted(absent)}")
        raw["cohort"] = raw["Group"].map(_clean_group_name)
        selected = raw[raw["cohort"].isin(PATHWAY_COHORTS)].copy()
        choices.append((selected["cohort"].nunique(), len(selected), path, selected))
    if not choices:
        raise FileNotFoundError(f"No usable {source} enrichment table in {enrichment_dir}")
    _, _, path, selected = max(choices, key=lambda x: (x[0], x[1]))
    missing = sorted(set(PATHWAY_COHORTS).difference(selected["cohort"]))
    if missing:
        warnings.warn(f"{source}: selected table {path.name} is missing {missing}")
    selected = selected.rename(columns={"GO_Term": "pathway_name", "Term": "library"})
    selected["source"] = source
    selected["source_file"] = str(path)
    selected["neg_log10_p"] = pd.to_numeric(selected["P_value"], errors="coerce")
    selected["E_ratio"] = pd.to_numeric(selected["E_ratio"], errors="coerce")
    selected["p_value_raw"] = 10.0 ** (-selected["neg_log10_p"])
    selected["pathway_key"] = selected["source"] + "|" + selected["ID"].astype(str) + "|" + selected["pathway_name"].astype(str)
    columns = ["source", "source_file", "cohort", "Group", "ID", "pathway_name", "library", "m", "M", "n", "N", "E_ratio", "neg_log10_p", "p_value_raw", "pathway_key"]
    return selected[columns]


def _text_cluster(all_pathways):
    """Reproduce legacy TF-IDF/LSA/K-means clustering with fixed seed and n_init."""
    from sklearn.cluster import KMeans
    from sklearn.decomposition import TruncatedSVD
    from sklearn.feature_extraction.text import ENGLISH_STOP_WORDS, TfidfVectorizer
    from sklearn.metrics import silhouette_score
    from sklearn.preprocessing import Normalizer

    candidate_rows = all_pathways[all_pathways["neg_log10_p"] >= 2.0].copy()
    terms = candidate_rows.groupby(["source", "ID", "pathway_name", "pathway_key"], as_index=False).agg(
        max_neg_log10_p=("neg_log10_p", "max"), mean_eratio=("E_ratio", "mean"), n_cohorts=("cohort", "nunique")
    ).reset_index(drop=True)
    if len(terms) < 7:
        raise ValueError("At least seven p <= 0.01 unique pathway terms are required for legacy clustering.")
    vectorizer = TfidfVectorizer(lowercase=True, stop_words=sorted(set(ENGLISH_STOP_WORDS).union(_EXTRA_STOPWORDS)), ngram_range=(1, 2), min_df=2, max_df=0.80, token_pattern=r"(?u)\b[A-Za-z][A-Za-z0-9-]+\b")
    matrix = vectorizer.fit_transform(terms["pathway_name"].fillna(""))
    if matrix.shape[1] < 3:
        raise ValueError("Insufficient repeated pathway text vocabulary for TF-IDF/LSA clustering.")
    n_components = max(2, min(50, matrix.shape[1] - 1, len(terms) - 1))
    lsa = Normalizer(copy=False).fit_transform(TruncatedSVD(n_components=n_components, random_state=7).fit_transform(matrix))
    max_k = min(18, max(6, len(terms) // 25))
    candidate_ks = [k for k in range(6, max_k + 1) if k < len(terms)]
    # sklearn KMeans does not expose n_jobs.  Limit BLAS/OpenMP threads when
    # threadpoolctl is present, retaining exactly the legacy algorithm/seed.
    try:
        from threadpoolctl import threadpool_limits
    except ImportError:
        from contextlib import nullcontext
        threadpool_limits = lambda limits: nullcontext()
    scan_rows = []
    with threadpool_limits(limits=1):
        for k in candidate_ks:
            labels = KMeans(n_clusters=k, n_init=50, random_state=7).fit_predict(lsa)
            scan_rows.append({"k": k, "silhouette_cosine": silhouette_score(lsa, labels, metric="cosine")})
        scan = pd.DataFrame(scan_rows)
        best_k = int(scan.sort_values("silhouette_cosine", ascending=False).iloc[0]["k"])
        terms["cluster_id"] = KMeans(n_clusters=best_k, n_init=100, random_state=7).fit_predict(lsa)
    features = np.asarray(vectorizer.get_feature_names_out())
    labels = []
    for cluster_id in sorted(terms["cluster_id"].unique()):
        positions = np.where(terms["cluster_id"].to_numpy() == cluster_id)[0]
        ordered = np.asarray(matrix[positions].mean(axis=0)).ravel().argsort()[::-1]
        top_features = []
        for position in ordered:
            feature = features[position]
            if feature and feature not in _EXTRA_STOPWORDS and not any(feature in old or old in feature for old in top_features):
                top_features.append(feature)
            if len(top_features) == 5:
                break
        labels.append({"cluster_id": cluster_id, "cluster_label": "; ".join(top_features) if top_features else f"cluster_{cluster_id}"})
    terms = terms.merge(pd.DataFrame(labels), on="cluster_id", how="left")
    summary = terms.groupby(["cluster_id", "cluster_label"], as_index=False).agg(
        n_terms=("pathway_key", "nunique"), n_go=("source", lambda x: int((x == "GO").sum())), n_reactome=("source", lambda x: int((x == "Reactome").sum())), n_kegg=("source", lambda x: int((x == "KEGG").sum())), max_neg_log10_p=("max_neg_log10_p", "max"), median_neg_log10_p=("max_neg_log10_p", "median")
    ).sort_values(["max_neg_log10_p", "n_terms"], ascending=[False, False])
    return terms, scan, summary


def _matched_domains(term):
    return [label for label, pattern in PATHWAY_DOMAINS.values() if re.search(pattern, str(term), flags=re.I)]


def build_pathway_sources(data_dir=DATA_DIR, table_dir=TABLE_DIR):
    """Load prior enrichment outputs, recompute post-processing, and save source tables."""
    data_dir, table_dir = Path(data_dir), Path(table_dir)
    table_dir.mkdir(parents=True, exist_ok=True)
    enrichment_dir = data_dir / "enrichment"
    all_pathways = pd.concat([_load_one_pathway_source(src, paths, enrichment_dir) for src, paths in PATHWAY_SOURCES.items()], ignore_index=True)
    terms, scan, cluster_summary = _text_cluster(all_pathways)
    clustered = all_pathways.merge(terms[["pathway_key", "cluster_id", "cluster_label"]], on="pathway_key", how="left")
    clustered["highlight_domains"] = clustered["pathway_name"].map(_matched_domains)
    domains = clustered[clustered["highlight_domains"].map(bool)].explode("highlight_domains").rename(columns={"highlight_domains": "domain"}).copy()
    domains["is_crosscheck_signal"] = domains["neg_log10_p"] >= -np.log10(0.05)
    domains["is_primary_signal"] = domains["neg_log10_p"] >= 2.0
    domains["is_strict_signal"] = domains["neg_log10_p"] >= 2.35
    crosscheck = domains[domains["is_crosscheck_signal"]].copy()
    primary = domains[domains["is_primary_signal"]].copy()
    cross_summary = crosscheck.groupby(["domain", "cohort", "source"], as_index=False).agg(n_terms=("pathway_key", "nunique"), max_neg_log10_p=("neg_log10_p", "max"), median_neg_log10_p=("neg_log10_p", "median"), max_eratio=("E_ratio", "max"))
    threshold_summaries = []
    for threshold_name, threshold in (("nominal_p_0.05", -np.log10(0.05)), ("primary_p_0.01", 2.0), ("strict_p_0.0045", 2.35), ("very_strict_p_0.001", 3.0)):
        summary = domains[domains["neg_log10_p"] >= threshold].groupby(["domain", "source", "cohort"], as_index=False).agg(n_terms=("pathway_key", "nunique"), max_neg_log10_p=("neg_log10_p", "max"))
        summary["threshold"] = threshold_name
        summary["neg_log10_cutoff"] = threshold
        threshold_summaries.append(summary)
    threshold_summary = pd.concat(threshold_summaries, ignore_index=True)
    source_counts = cross_summary.pivot_table(index=["domain", "cohort"], columns="source", values="n_terms", fill_value=0, aggfunc="sum").reset_index()
    for source in ("GO", "Reactome", "KEGG"):
        if source not in source_counts:
            source_counts[source] = 0
    source_counts["reactome_or_kegg"] = source_counts["Reactome"] + source_counts["KEGG"]
    source_counts["cross_checked_by_reactome_or_kegg"] = source_counts["reactome_or_kegg"] > 0
    cluster_domain_summary = primary.groupby(["domain", "cluster_id", "cluster_label", "source", "cohort"], dropna=False, as_index=False).agg(n_terms=("pathway_key", "nunique"), max_neg_log10_p=("neg_log10_p", "max"), max_eratio=("E_ratio", "max")).sort_values(["domain", "max_neg_log10_p"], ascending=[True, False])
    pathway_records = []
    for (domain, source, pathway_name, pathway_key), group in domains.groupby(["domain", "source", "pathway_name", "pathway_key"], dropna=False):
        passing_primary = group[group["is_primary_signal"]]
        pathway_records.append({"domain": domain, "source": source, "pathway_name": pathway_name, "pathway_key": pathway_key, "max_neg_log10_p": group["neg_log10_p"].max(), "max_E_ratio": group["E_ratio"].max(), "n_cohorts_any": group["cohort"].nunique(), "n_cohorts_primary": passing_primary["cohort"].nunique(), "cohorts_primary": ", ".join(sorted(passing_primary["cohort"].unique()))})
    pathways_per_domain = pd.DataFrame(pathway_records).sort_values(["domain", "source", "max_neg_log10_p"], ascending=[True, True, False])
    # Figure 3A: GO only, four specified domains, nominal p <= 0.05.
    figure3a = cross_summary[cross_summary["source"].eq("GO") & cross_summary["domain"].isin(FIG3A_DOMAIN_ORDER)].copy()
    figure3a = figure3a.set_index(["domain", "cohort"]).reindex(pd.MultiIndex.from_product([FIG3A_DOMAIN_ORDER, PATHWAY_COHORTS], names=["domain", "cohort"])).reset_index()
    figure3a["source"] = "GO"
    figure3a[["n_terms", "max_neg_log10_p", "median_neg_log10_p", "max_eratio"]] = figure3a[["n_terms", "max_neg_log10_p", "median_neg_log10_p", "max_eratio"]].fillna(0)
    figure3a["n_terms"] = figure3a["n_terms"].astype(int)
    # S7A: choose up to four terms separately for every domain/source/tumor, then retain GTEx matching points.
    tumors = primary[primary["cohort"].isin(("CPTAC-3_Tumor", "TCGA-PAAD_Tumor"))].copy()
    tumor_top = tumors.sort_values(["domain", "source", "cohort", "neg_log10_p"], ascending=[True, True, True, False]).groupby(["domain", "source", "cohort"], group_keys=False).head(4)
    selected = set(zip(tumor_top["domain"], tumor_top["source"], tumor_top["pathway_name"]))
    dotplot = primary[primary["cohort"].isin(PATHWAY_COHORTS)].copy()
    dotplot = dotplot[[(d, s, n) in selected for d, s, n in zip(dotplot["domain"], dotplot["source"], dotplot["pathway_name"])]].copy()
    term_order = tumor_top.sort_values(["domain", "source", "neg_log10_p"], ascending=[True, True, False]).apply(lambda row: (row["domain"], row["source"], row["pathway_name"]), axis=1).drop_duplicates().tolist()
    order_map = {key: i for i, key in enumerate(term_order)}
    dotplot["term_order"] = [order_map[(d, s, n)] for d, s, n in zip(dotplot["domain"], dotplot["source"], dotplot["pathway_name"])]
    dotplot["display_label"] = dotplot.apply(lambda row: f"{row['domain']} · {row['source']}\n" + "\n".join(textwrap.wrap(str(row["pathway_name"]), width=50, break_long_words=False)), axis=1)
    dotplot["cohort"] = pd.Categorical(dotplot["cohort"], PATHWAY_COHORTS, ordered=True)
    # Tables make every input choice and every panel cell auditable.
    all_pathways.to_csv(table_dir / "pathway_grouping_input_all_sources.tsv", sep="\t", index=False)
    scan.to_csv(table_dir / "pathway_text_cluster_k_scan.tsv", sep="\t", index=False)
    terms.to_csv(table_dir / "pathway_text_clusters.tsv", sep="\t", index=False)
    cluster_summary.to_csv(table_dir / "pathway_text_cluster_summary.tsv", sep="\t", index=False)
    crosscheck.to_csv(table_dir / "highlighted_domain_pathways_crosscheck_threshold.tsv", sep="\t", index=False)
    primary.to_csv(table_dir / "highlighted_domain_pathways_primary_threshold.tsv", sep="\t", index=False)
    cross_summary.to_csv(table_dir / "domain_crosscheck_summary.tsv", sep="\t", index=False)
    threshold_summary.to_csv(table_dir / "domain_threshold_sensitivity.tsv", sep="\t", index=False)
    source_counts.to_csv(table_dir / "domain_crosscheck_source_counts.tsv", sep="\t", index=False)
    cluster_domain_summary.to_csv(table_dir / "cluster_domain_summary.tsv", sep="\t", index=False)
    pathways_per_domain.to_csv(table_dir / "pathways_per_domain.tsv", sep="\t", index=False)
    figure3a.to_csv(table_dir / "Figure3A_GO_domain_heatmap_source.tsv", sep="\t", index=False)
    dotplot.sort_values(["term_order", "cohort"]).to_csv(table_dir / "FigureS7A_highlighted_pathway_dotplot_source.tsv", sep="\t", index=False)
    return {"all_pathways": all_pathways, "clusters": terms, "domain_rows": domains, "figure3a": figure3a, "s7a": dotplot, "s7a_terms": term_order}


def draw_figure3a_pathways(subfig, sources=None):
    """Draw the GO-domain heatmap in one square publication module."""
    sources = PATHWAY_SOURCES_DATA if sources is None else sources
    source = sources["figure3a"]
    values = source.pivot(index="domain", columns="cohort", values="max_neg_log10_p").reindex(index=FIG3A_DOMAIN_ORDER, columns=PATHWAY_COHORTS).fillna(0)
    counts = source.pivot(index="domain", columns="cohort", values="n_terms").reindex(index=FIG3A_DOMAIN_ORDER, columns=PATHWAY_COHORTS).fillna(0).astype(int)
    # Fixed coordinates reserve space for full domain names while giving the
    # heatmap most of the remaining width; the color scale sits below it.
    ax = subfig.add_axes([.37, .32, .59, .53])
    ax.set_in_layout(False)
    im = ax.imshow(values.to_numpy(), cmap="YlOrRd", vmin=0, aspect="auto")
    white_text_cutoff = 0.60 * float(values.to_numpy().max())
    for y, domain in enumerate(values.index):
        for x, cohort in enumerate(values.columns):
            ax.text(x, y, str(counts.loc[domain, cohort]), ha="center", va="center",
                    fontsize=FONT["annotation"], fontweight="normal",
                    color="white" if values.loc[domain, cohort] > white_text_cutoff else "black")
    ax.set_xticks(range(3), ["GTEx", "CPTAC", "TCGA"], fontsize=FONT["tick"])
    ax.set_yticks(range(4), ["Splicing", "APA", "miRNA", "Transcription"], fontsize=FONT["tick"])
    ax.tick_params(axis="both", length=0, pad=3)
    for spine in ax.spines.values(): spine.set_visible(False)
    subfig.text(.5, .995, "GO domains", ha="center", va="top",
                fontsize=FONT["title"], fontweight="bold")
    subfig.text(.5, .92, "P ≤ .05; term counts", ha="center", va="top",
                fontsize=FONT["annotation"], fontweight="normal")
    cax = subfig.add_axes([.37, .12, .59, .035])
    cax.set_in_layout(False)
    colorbar = subfig.colorbar(im, cax=cax, orientation="horizontal")
    colorbar.set_ticks([0, 10, 20])
    colorbar.set_label("max −log₁₀(P)", fontsize=FONT["legend"], labelpad=3)
    colorbar.ax.xaxis.set_label_position("top")
    colorbar.ax.tick_params(labelsize=FONT["tick"], length=2, pad=2)
    colorbar.outline.set_linewidth(.5)
    return ax


def draw_s7a_pathways(subfig, sources=None):
    """Draw S7A highlighted term dotplot into a 4 × 6 module SubFigure."""
    sources = PATHWAY_SOURCES_DATA if sources is None else sources
    data, terms = sources["s7a"].copy(), sources["s7a_terms"]
    # The dedicated left margin preserves readable, wrapped full term labels;
    # no exporter-side tight crop is needed.
    ax = subfig.subplots()
    y_positions = {key: i for i, key in enumerate(terms)}
    shapes = {"GO": "o", "Reactome": "s", "KEGG": "^"}
    size_min, size_max = 24, 125
    finite = data["E_ratio"].replace([np.inf, -np.inf], np.nan).dropna()
    low, high = (finite.min(), finite.max()) if len(finite) else (0, 1)
    scale = lambda value: size_min + (size_max - size_min) * ((value - low) / (high - low) if high > low else 0.5)
    from matplotlib.colors import Normalize
    normalization = Normalize(vmin=max(2.0, data["neg_log10_p"].min()), vmax=data["neg_log10_p"].max())
    mapper = plt.cm.ScalarMappable(norm=normalization, cmap="viridis")
    for source, shape in shapes.items():
        part = data[data["source"].eq(source)]
        ax.scatter(part["cohort"].cat.codes, [y_positions[(r.domain, r.source, r.pathway_name)] for r in part.itertuples()], s=[scale(v) for v in part["E_ratio"]], c=part["neg_log10_p"], cmap="viridis", norm=normalization, marker=shape, edgecolor="black", linewidth=0.25, label=source)
    labels = []
    for domain, source, name in terms:
        candidates = data[(data["domain"] == domain) & (data["source"] == source) & (data["pathway_name"] == name)]
        labels.append(candidates["display_label"].iloc[0])
    ax.set_yticks(range(len(terms)), labels, fontsize=FONT["tick"])
    ax.set_xticks(range(3), ["GTEx", "CPTAC", "TCGA"], fontsize=FONT["tick"])
    ax.set_xlabel("Cohort", fontsize=FONT["axis_title"])
    ax.set_ylabel("Highlighted pathway", fontsize=FONT["axis_title"])
    ax.set_title("Top tumor pathways by domain and library\n(p ≤ 0.01; GTEx shown when matching)", fontsize=FONT["title"], loc="left")
    ax.invert_yaxis()
    ax.grid(axis="x", color="#D9D9D9", linewidth=0.5)
    ax.set_axisbelow(True)
    colorbar = subfig.colorbar(mapper, ax=ax, fraction=0.018, pad=0.015)
    colorbar.set_label("−log10(p)", fontsize=FONT["legend"])
    colorbar.ax.tick_params(labelsize=FONT["tick"])
    handles, labels = ax.get_legend_handles_labels()
    library_legend = ax.legend(handles, labels, title="Library", loc="upper left", bbox_to_anchor=(1.10, 1), frameon=False, fontsize=FONT["legend"], title_fontsize=FONT["legend"])
    ax.add_artist(library_legend)
    size_values = np.unique(np.round(np.quantile(finite, [0.1, 0.5, 0.9]), 2))
    size_handles = [ax.scatter([], [], s=scale(value), facecolor="white", edgecolor="black", linewidth=0.25) for value in size_values]
    ax.legend(size_handles, [f"{value:g}" for value in size_values], title="Enrichment ratio", loc="upper left", bbox_to_anchor=(1.10, 0.73), frameon=False, fontsize=FONT["legend"], title_fontsize=FONT["legend"])
    return ax


PATHWAY_SOURCES_DATA = build_pathway_sources()
PANELS["3A"] = {"draw": draw_figure3a_pathways, "content": "go_pathway_domain_heatmap", "width": 2, "height": 2}
PANELS["S7A"] = {"draw": draw_s7a_pathways, "content": "highlighted_pathway_dotplot", "width": 4, "height": 6}


## 4. Shared figure functions

Individual panels and assemblies use the same drawing functions. Assemblies retain
native panel sizes and font sizes; PDF content remains vector-based apart from
intentional heatmap rasterization. Long pathway lists and the 81-gene machinery
heatmap use taller layouts to preserve readable text.

In [ ]:
# Cell 7: Shared panel drawing functions
"""Vector-native plotting helpers for Figure 3 and Supplementary Figures S7--S9.

Every public draw function accepts a matplotlib ``SubFigure`` and the analysis
``DATA`` mapping.  The registry is deliberately data-free, so the notebook can
export a panel by itself or place the same callable in an assembled figure.
"""
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from pathlib import Path
from matplotlib.colors import TwoSlopeNorm
from matplotlib.patches import Arc, Rectangle
from scipy import stats

try:  # The notebook setup owns these shared publication globals.
    FONT
except NameError:
    FONT = dict(base=8, title=9, axis_title=8, tick=7, legend=7, annotation=7, tag=10)
try:
    PANEL_MM
except NameError:
    PANEL_MM = 52.5
try:
    COHORT_INFO
except NameError:
    COHORT_INFO = {
        "CPTAC_Tumor": {"label": "CPTAC tumor", "colour": "#0072B2"},
        "TCGA_Tumor": {"label": "TCGA tumor", "colour": "#D55E00"},
        "GTEx_Normal": {"label": "GTEx normal", "colour": "#009E73"},
    }
# These encode data type, rather than cohort identity, in the phase comparisons.
AS_COLOR = "#E69F00"
EXPR_COLOR = "#0072B2"
PEAK_COLOR = "#C51B7D"
NO_PEAK_COLOR = "#4D4D4D"


def _cohort_label(cohort):
    return COHORT_INFO[cohort].get("label", cohort)


def _cohort_colour(cohort):
    info = COHORT_INFO[cohort]
    return info.get("colour", info.get("color", "#666666"))


def _data(data):
    if data is None:
        try:
            return DATA  # noqa: F821 - supplied by the publication notebook
        except NameError as exc:
            raise ValueError("Pass the DATA mapping to the panel draw function.") from exc
    return data


def _axes(subfig, *, polar=False):
    ax = subfig.add_subplot(111, projection="polar" if polar else None)
    return ax


def _style(ax, xlabel=None, ylabel=None, title=None):
    if xlabel:
        ax.set_xlabel(xlabel, fontsize=FONT["axis_title"])
    if ylabel:
        ax.set_ylabel(ylabel, fontsize=FONT["axis_title"])
    if title:
        ax.set_title(title, fontsize=FONT["title"], fontweight="bold", pad=4)
    ax.tick_params(labelsize=FONT["tick"])
    if not getattr(ax, "name", "") == "polar":
        ax.spines[["top", "right"]].set_visible(False)


def _phase_hours(x):
    """Input contract: all publication DATA acrophases are in circadian hours."""
    return np.asarray(x, dtype=float) % 24


def _theta(x):
    return _phase_hours(x) * 2 * np.pi / 24


def _polar_clock(ax):
    ax.set_theta_zero_location("N")
    ax.set_theta_direction(-1)
    ticks = np.arange(0, 24, 4)
    ax.set_xticks(ticks * 2 * np.pi / 24)
    ax.set_xticklabels([str(x) for x in ticks], fontsize=FONT["tick"])
    ax.tick_params(axis="y", labelsize=FONT["tick"])


def _kde(theta, bandwidth=None, grid_n=360):
    bandwidth = CONFIG["circular_bandwidth"] if bandwidth is None else bandwidth
    theta = np.asarray(theta, dtype=float)
    theta = theta[np.isfinite(theta)] % (2 * np.pi)
    grid = np.linspace(0, 2 * np.pi, grid_n, endpoint=True)
    if not len(theta):
        return grid, np.zeros_like(grid)
    distance = np.angle(np.exp(1j * (grid[:, None] - theta[None, :])))
    # Von Mises KDE.  The subtraction preserves numerical stability and has no
    # effect after normalisation; bandwidth .30 maps to kappa=1/bandwidth².
    density = np.exp((np.cos(distance) - 1) / bandwidth ** 2).sum(axis=1)
    density /= np.trapezoid(density, grid)
    return grid, density


def _axial_orientation(theta):
    theta = np.asarray(theta, dtype=float)
    theta = theta[np.isfinite(theta)]
    if not len(theta):
        return np.nan, np.nan
    z = np.mean(np.exp(2j * theta))
    r2 = abs(z)
    hour = ((np.angle(z) / 2) % np.pi) * 12 / np.pi
    return r2, hour


def _event_frame(data, cohort):
    frame = _data(data)["events"][cohort].copy()
    out = frame.loc[frame["event_type"].astype(str).eq("SE")].copy()
    if "p" not in out:
        raise ValueError(f"{cohort} events require nominal p values for the legacy p < 0.05 call.")
    return out.loc[pd.to_numeric(out["p"], errors="coerce") < CONFIG["as_p"]].copy()


def _acrophase_hours(frame):
    """Return the explicit-hour acrophase column from a contract table."""
    for col in ("acrophase_hours", "acrophase"):
        if col in frame:
            return pd.to_numeric(frame[col], errors="coerce").to_numpy() % 24
    raise ValueError("Expected acrophase_hours (or legacy acrophase in hours).")


def _phase_density(subfig, data, cohort):
    frame = _event_frame(data, cohort)
    if frame.empty:
        raise ValueError(f"No SE events available for {cohort}.")
    theta = _theta(_acrophase_hours(frame))
    grid, density = _kde(theta)
    r2, axis_h = _axial_orientation(theta)
    ax = _axes(subfig, polar=True)
    color = _cohort_colour(cohort)
    ax.plot(grid, density, color=color, lw=1.5)
    ax.fill_between(grid, 0, density, color=color, alpha=.25)
    if np.isfinite(axis_h):
        for hour, alpha in ((axis_h, 1), ((axis_h + 12) % 24, .45)):
            ax.axvline(hour * 2 * np.pi / 24, color="#333333", ls="--", lw=.8, alpha=alpha)
    _polar_clock(ax)
    ax.set_title(f"{_cohort_label(cohort)}\nSE phase; n={len(frame)}\naxial R₂={r2:.2f}",
                 fontsize=FONT["title"], fontweight="bold", pad=5)
    return ax


def draw_3b(subfig, data=None): return _phase_density(subfig, data, "GTEx_Normal")
def draw_3c(subfig, data=None): return _phase_density(subfig, data, "TCGA_Tumor")


def draw_3d(subfig, data=None):
    slk = _data(data)["slk"]
    samples = slk["samples"].copy()
    reps = slk.get("representatives", pd.DataFrame()).copy()
    required = {"phase_h", "psi"}
    if not required.issubset(samples): raise ValueError("SLK samples need phase_h and psi columns.")
    ax = _axes(subfig)
    ax.scatter(samples.phase_h, samples.psi, s=9, c=".70", alpha=.65, linewidths=0, label="all samples")
    fit_phase = np.asarray(slk.get("fit_phase", []), dtype=float)
    fit_psi = np.asarray(slk.get("fit_psi", []), dtype=float)
    if len(fit_phase) != len(fit_psi) or not len(fit_phase):
        raise ValueError("SLK fit_phase and fit_psi arrays are required for the harmonic fit.")
    order = np.argsort(fit_phase)
    ax.plot(fit_phase[order], fit_psi[order], c=AS_COLOR, lw=1.3, label="harmonic fit")
    if not reps.empty:
        ax.scatter(reps.phase_h, reps.psi, s=28, c=np.arange(len(reps)), cmap="viridis", edgecolor="black", lw=.35,
                   zorder=3, label="representatives")
        for row in reps.itertuples():
            ax.annotate(f"{row.phase_h:.1f} h", (row.phase_h, row.psi),
                        xytext=(2, 3), textcoords="offset points", fontsize=FONT["annotation"])
    ax.set_xlim(0, 24); ax.set_xticks(np.arange(0, 25, 6))
    _style(ax, "Circadian phase (h)", "PSI", "SLK skipped-exon rhythm")
    ax.legend(frameon=False, fontsize=FONT["legend"], loc="best")
    return ax


# SLK event 150506 coordinates from the processed rMATS SE table (0-based,
# half-open); exon models are event inclusion/skipping paths, not full transcripts.
SLK_BAM_SETTINGS = dict(event_id='150506', gene='SLK', contig='chr10', strand='+',
                        genome='GRCh38', chr10_length=133797422,
                        exons=[[104008176,104008356],[104010815,104010908],[104018159,104018289]],
                        flank_bp=100, intron_scale=5, read_length=48,
                        filter='primary, QC-pass, proper pair, NH=1, no soft/hard clipping',
                        depth_unit='aligned reads per base; overlapping mates counted separately')

def build_slk_bam_sources(representatives, rmats_counts, bam_dir=None, output_dir=None, allow_cache=None):
    """Extract real BAM coverage/junctions, or validate a portable extracted bundle.

    Complete indexed BAMs are preferred. The portable bundle permits figure-only
    regeneration without distributing controlled-access individual alignments.
    No smoothing, interpolation of read depth, or synthetic coverage is used.
    """
    import hashlib, json, os
    from collections import Counter
    bam_dir = Path(bam_dir) if bam_dir is not None else None
    allow_cache = SLK_USE_SOURCE_CACHE if allow_cache is None else allow_cache
    output_dir = Path(output_dir or TABLE_DIR)
    output_dir.mkdir(parents=True,exist_ok=True)
    settings = SLK_BAM_SETTINGS
    samples = representatives[['sample_id','phase_h','psi']].sort_values('phase_h').reset_index(drop=True)
    expected = rmats_counts.set_index('sample_id').loc[samples.sample_id,['UTC','TDC','UDC']].astype(int)
    names = {key:f'Figure3E_SLK_bam_{key}.tsv' for key in ['coverage','junctions','qc','exons','samples']}
    manifest_path = output_dir/'Figure3E_SLK_bam_provenance.json'
    paths = [bam_dir/f'{sample}.bam' if bam_dir is not None else Path(SLK_BAM_PATHS[sample])
             for sample in samples.sample_id]
    def digest(path):
        h=hashlib.sha256()
        with Path(path).open('rb') as stream:
            for block in iter(lambda:stream.read(8*1024*1024),b''): h.update(block)
        return h.hexdigest()
    if not all(path.is_file() for path in paths):
        if not allow_cache or not manifest_path.is_file():
            raise FileNotFoundError('Supply four indexed BAMs via CHRONO_SLK_BAM_DIR or the Figure3E_SLK_bam source bundle.')
        meta=json.loads(manifest_path.read_text())
        if meta['settings']!=settings: raise ValueError('SLK BAM bundle settings differ; regenerate it from BAMs.')
        for filename, checksum in meta['table_sha256'].items():
            if digest(output_dir/filename)!=checksum: raise ValueError(f'SLK bundle checksum mismatch: {filename}')
        bundle={k:pd.read_csv(output_dir/v,sep='\t') for k,v in names.items()}
        got=bundle['samples'].set_index('sample_id').loc[samples.sample_id]
        if not np.allclose(got[['phase_h','psi']],samples[['phase_h','psi']]):
            raise ValueError('SLK BAM bundle representative phases/PSI differ from the current analysis.')
        observed=bundle['junctions'].pivot(index='sample_id',columns='junction',values='bam_read_count').reindex(index=expected.index,columns=expected.columns)
        if not observed.equals(expected):
            raise ValueError('SLK BAM junction counts differ from the current rMATS records.')
        bundle['provenance']=meta
        return bundle
    import pysam
    exon_array=np.asarray(settings['exons'],int)
    start=int(exon_array[0,0]-settings['flank_bp']); stop=int(exon_array[-1,1]+settings['flank_bp'])
    targets={(int(exon_array[0,1]),int(exon_array[1,0])):'UTC',
             (int(exon_array[1,1]),int(exon_array[2,0])):'TDC',
             (int(exon_array[0,1]),int(exon_array[2,0])):'UDC'}
    coverage=[]; junctions=[]; qc=[]; inputs=[]
    for sample,path in zip(samples.sample_id,paths):
        pysam.quickcheck(str(path))
        with pysam.AlignmentFile(str(path),'rb') as bam:
            if not bam.has_index(): raise ValueError(f'BAM index is required: {path.name}')
            header=bam.header.to_dict()
            if header.get('HD',{}).get('SO')!='coordinate': raise ValueError('Coordinate-sorted BAM required.')
            if bam.get_reference_length(settings['contig'])!=settings['chr10_length']:
                raise ValueError(f'GRCh38 chr10 length mismatch: {path.name}')
            rg_samples={r.get('SM','') for r in header.get('RG',[])}
            if not rg_samples or not all(s.startswith(sample) for s in rg_samples):
                raise ValueError(f'BAM read-group sample mismatch: {sample}, {rg_samples}')
            depth=np.zeros(stop-start,dtype=np.int64); counts=Counter(); seen=0; used=0; read_lengths=Counter()
            for read in bam.fetch(settings['contig'],start,stop):
                seen+=1
                if (read.is_unmapped or read.is_secondary or read.is_supplementary or read.is_qcfail
                    or not read.is_proper_pair or not read.has_tag('NH') or read.get_tag('NH')!=1
                    or any(op in (4,5) for op,length in (read.cigartuples or []))): continue
                if read.query_length!=settings['read_length']:
                    raise ValueError(f'Unexpected read length in {sample}: {read.query_length}')
                used+=1;read_lengths[read.query_length]+=1;position=read.reference_start
                for op,length in read.cigartuples or []:
                    if op in (0,7,8):
                        a=max(position,start);b=min(position+length,stop)
                        if b>a: depth[a-start:b-start]+=1
                    elif op==3 and (position,position+length) in targets:
                        counts[targets[(position,position+length)]]+=1
                    if op in (0,2,3,7,8): position+=length
            coverage.append(pd.DataFrame(dict(sample_id=sample,position_0based=np.arange(start,stop),depth=depth)))
            for (donor,acceptor),junction in targets.items():
                observed=int(counts[junction]); reference=int(expected.loc[sample,junction])
                junctions.append(dict(sample_id=sample,junction=junction,donor_0based=donor,
                                      acceptor_0based=acceptor,bam_read_count=observed,rmats_read_count=reference,
                                      matches_rmats=observed==reference))
                if observed!=reference: raise ValueError(f'{sample} {junction}: BAM {observed} != rMATS {reference}')
            qc.append(dict(sample_id=sample,region_alignments=seen,retained_alignments=used,
                           mapped_alignments_whole_bam=int(bam.mapped),max_depth=int(depth.max()),
                           read_length=settings['read_length'],bam_quickcheck='pass',sample_header_check='pass',
                           genome_coordinate_check='pass',all_three_junction_counts_match=True))
            index=next((p for p in [Path(str(path)+'.bai'),path.with_suffix('.bai'),Path(str(path)+'.csi')] if p.exists()),None)
            inputs.append(dict(sample_id=sample,bam_name=path.name,bytes=path.stat().st_size,
                               mtime_ns=path.stat().st_mtime_ns,read_group_samples=sorted(rg_samples),
                               header_sha256=hashlib.sha256(str(bam.header).encode()).hexdigest(),
                               index_name=index.name if index else None,index_sha256=digest(index) if index else None))
    exons=pd.DataFrame([dict(exon=label,start_0based=int(a),end_0based=int(b))
                        for label,(a,b) in zip(['upstream','skipped','downstream'],exon_array)])
    bundle=dict(coverage=pd.concat(coverage,ignore_index=True),junctions=pd.DataFrame(junctions),
                qc=pd.DataFrame(qc),exons=exons,samples=samples)
    for key,filename in names.items():bundle[key].to_csv(output_dir/filename,sep='\t',index=False)
    meta=dict(settings=settings,inputs=inputs,pysam_version=pysam.__version__,
              samtools_version=pysam.__samtools_version__,coordinates='0-based half-open; plot labels use 1-based coordinates',
              bam_full_file_sha256='not computed; BAM header and index hashes plus size/mtime recorded',
              table_sha256={filename:digest(output_dir/filename) for filename in names.values()})
    manifest_path.write_text(json.dumps(meta,indent=2)+'\n');bundle['provenance']=meta
    return bundle

def draw_3e(subfig, data=None):
    """True read depth and measured splice junctions from the four selected BAMs."""
    from matplotlib.path import Path as MplPath
    from matplotlib.patches import PathPatch
    from matplotlib.ticker import MaxNLocator
    d=_data(data)
    if 'slk_bam' not in d:
        d['slk_bam']=build_slk_bam_sources(d['slk']['representatives'],d['slk']['junction_counts'])
    bundle=d['slk_bam'];exons=bundle['exons'];params=bundle['provenance']['settings']
    intervals=exons[['start_0based','end_0based']].to_numpy(int)
    start=intervals[0,0]-params['flank_bp'];end=intervals[-1,1]+params['flank_bp']
    knots=np.array([start,*intervals.ravel(),end],float)
    scales=np.array([1,1,params['intron_scale'],1,params['intron_scale'],1,1],float)
    display_knots=np.r_[0,np.cumsum(np.diff(knots)/scales)]
    project=lambda x:np.interp(x,knots,display_knots)
    colour='#CC001B';axes=[]
    subfig.text(.5,.997,'SLK skipped-exon event 150506',ha='center',va='top',
                fontsize=FONT['title'],fontweight='bold')
    for i,row in enumerate(bundle['samples'].itertuples()):
        y=.755-i*.18
        ax=subfig.add_axes([.075,y,.91,.128]);ax.set_in_layout(False);axes.append(ax)
        cov=bundle['coverage'].loc[bundle['coverage'].sample_id.eq(row.sample_id)]
        peak=max(1,float(cov.depth.max()));x=project(cov.position_0based.to_numpy()+.5)
        ax.fill_between(x,0,cov.depth.to_numpy(),step='mid',color=colour,lw=0)
        ax.set_xlim(display_knots[0],display_knots[-1]);ax.set_ylim(-.65*peak,1.75*peak)
        ax.set_yticks([0,int(peak)])
        ax.tick_params(axis='y',labelsize=FONT['tick'],length=2,pad=2)
        ax.set_xticks([]);ax.spines['left'].set_bounds(0,peak)
        for side in ['top','right','bottom']:ax.spines[side].set_visible(False)
        for record in bundle['junctions'].loc[bundle['junctions'].sample_id.eq(row.sample_id)].itertuples():
            x0,x1=project([record.donor_0based,record.acceptor_0based])
            # The first-to-third exon (skipping) junction is above coverage.
            arc_height=(1.4 if record.junction=='UDC' else -.32)*peak
            verts=[(x0,0),((x0+x1)/2,2*arc_height),(x1,0)]
            ax.add_patch(PathPatch(MplPath(verts,[MplPath.MOVETO,MplPath.CURVE3,MplPath.CURVE3]),
                         fill=False,color=colour,lw=.5+.20*np.log1p(record.bam_read_count)))
            ax.text((x0+x1)/2,arc_height,str(record.bam_read_count),ha='center',va='center',
                    fontsize=FONT['annotation'],fontweight='normal',
                    bbox=dict(facecolor='white',edgecolor='none',pad=.25))
        subfig.text(.985,y+.14,f'{row.sample_id}   phase {row.phase_h:.1f} h   PSI {row.psi:.3f}',
                    ha='right',va='bottom',fontsize=FONT['annotation'],fontweight='normal')
    subfig.text(.012,.56,'Read depth',rotation=90,ha='center',va='center',fontsize=FONT['axis_title'])
    model=subfig.add_axes([.075,.025,.91,.115]);model.set_in_layout(False)
    for ypos,indices in [(.70,[0,1,2]),(.30,[0,2])]:
        model.plot(project([intervals[0,0],intervals[-1,1]]),[ypos,ypos],color='black',lw=.5)
        for idx in indices:
            x0,x1=project(intervals[idx]);model.add_patch(Rectangle((x0,ypos-.08),x1-x0,.16,color='black'))
        for xpos in np.linspace(project(intervals[0,1])+15,project(intervals[-1,0])-15,28):
            if any(project(intervals[idx])[0]<=xpos<=project(intervals[idx])[1] for idx in indices):continue
            model.plot([xpos-4,xpos,xpos-4],[ypos-.025,ypos,ypos+.025],color='black',lw=.45)
    subfig.text(.075,.15,'SLK (+) • inclusion / skipping',ha='left',va='bottom',fontsize=FONT['annotation'])
    subfig.text(.985,.15,'GRCh38 chr10 • introns compressed 5×',ha='right',va='bottom',fontsize=FONT['annotation'])
    model.text(0,-.12,f'{intervals[0,0]+1:,}',ha='left',va='top',fontsize=FONT['tick'])
    model.text(display_knots[-1],-.12,f'{intervals[-1,1]:,}',ha='right',va='top',fontsize=FONT['tick'])
    model.set_xlim(display_knots[0],display_knots[-1]);model.set_ylim(-.25,1.1);model.set_axis_off()
    return axes[0]


def _draw_heatmap(subfig, data, cohort):
    hm = _data(data)["heatmaps"][cohort]
    matrix = np.asarray(hm["matrix"], dtype=float)
    if matrix.ndim != 2 or matrix.shape[1] != 24: raise ValueError(f"{cohort} heatmap must be events × 24 bins.")
    ax = _axes(subfig)
    im = ax.imshow(matrix, cmap="RdBu_r", norm=TwoSlopeNorm(vcenter=0, vmin=-2, vmax=2), aspect="auto", interpolation="nearest")
    ax.set_xticks(np.arange(0, 24, 4)); ax.set_xticklabels(np.arange(0, 24, 4))
    ax.set_yticks([])
    _style(ax, "Circadian phase (h)", "SE events (phase-sorted)", f"{_cohort_label(cohort)} (n={matrix.shape[0]})")
    cbar = subfig.colorbar(im, ax=ax, fraction=.045, pad=.02)
    cbar.set_label("PSI z-score", fontsize=FONT["axis_title"]); cbar.ax.tick_params(labelsize=FONT["tick"])
    return ax
def draw_3f(subfig, data=None): return _draw_heatmap(subfig, data, "TCGA_Tumor")
def draw_s7b(subfig, data=None): return _draw_heatmap(subfig, data, "CPTAC_Tumor")
def draw_s7c(subfig, data=None): return _draw_heatmap(subfig, data, "GTEx_Normal")


def _machinery_polar(subfig, data, cohort):
    m = _data(data)["machinery"].copy(); m = m.loc[m.cohort.eq(cohort) & (m.p < CONFIG["as_p"])]
    if m.empty: raise ValueError(f"No machinery results for {cohort}.")
    ax = _axes(subfig, polar=True)
    cats = list(_data(data).get("machinery_categories", {})) or list(pd.unique(m.category))
    cmap = plt.get_cmap("tab10")
    for i, cat in enumerate(cats):
        d = m.loc[m.category.eq(cat)]
        if d.empty: continue
        ax.scatter(_theta(_acrophase_hours(d)), d.amplitude, s=16, color=cmap(i), alpha=.85, label=str(cat), edgecolor="white", lw=.2)
    _polar_clock(ax); ax.set_title(f"{_cohort_label(cohort)} machinery\nraw p < {CONFIG['as_p']:g}; n={len(m)}", fontsize=FONT["title"], fontweight="bold", pad=5)
    for i, row in enumerate(m.nlargest(3, 'amplitude').itertuples()):
        left = row.acrophase > 12
        ax.annotate(row.gene_symbol, (row.acrophase*np.pi/12, row.amplitude),
                    xytext=(-3 if left else 3, 6 if i % 2 == 0 else -6),
                    ha='right' if left else 'left', textcoords='offset points', fontsize=FONT['annotation'])
    ax.legend(fontsize=FONT['legend'], frameon=False, loc='upper center', bbox_to_anchor=(.5,-.15))
    return ax
def draw_3g(subfig, data=None): return _machinery_polar(subfig, data, "TCGA_Tumor")
def draw_3h(subfig, data=None): return _machinery_polar(subfig, data, "GTEx_Normal")
def draw_s8a(subfig, data=None): return _machinery_polar(subfig, data, "CPTAC_Tumor")


def draw_3i(subfig, data=None):
    m = _data(data)["machinery"].copy(); cats = list(_data(data).get("machinery_categories", {})) or list(pd.unique(m.category))
    ax = _axes(subfig); x = np.arange(len(cats)); width=.36
    universe=m.groupby("category").gene_symbol.nunique().reindex(cats,fill_value=0)
    tumor=(m.loc[m.cohort.isin(["CPTAC_Tumor","TCGA_Tumor"]) & (pd.to_numeric(m.p,errors="coerce")<CONFIG["as_p"])]
             .groupby(["category","gene_symbol"]).size().reset_index().groupby("category").gene_symbol.nunique().reindex(cats,fill_value=0))
    normal=(m.loc[m.cohort.eq("GTEx_Normal") & (pd.to_numeric(m.p,errors="coerce")<CONFIG["as_p"])]
             .groupby("category").gene_symbol.nunique().reindex(cats,fill_value=0))
    ax.bar(x-width/2,100*tumor/universe, width, color="#D55E00", label="tumor (either cohort)")
    ax.bar(x+width/2,100*normal/universe, width, color="#009E73", label="GTEx normal")
    summary = []
    for i,c in enumerate(cats):
        n=int(universe[c]); _,pv=stats.fisher_exact([[int(tumor[c]),n-int(tumor[c])],[int(normal[c]),n-int(normal[c])]])
        if c == "Core Spliceosome" and pv < .05:
            ax.text(i,max(100*tumor[c]/n,100*normal[c]/n)+3,"*",ha="center",fontsize=FONT["annotation"])
        cohort_counts = {cohort: m.loc[m.category.eq(c) & m.cohort.eq(cohort) &
                         (pd.to_numeric(m.p,errors="coerce") < CONFIG["as_p"]),"gene_symbol"].nunique()
                         for cohort in ["TCGA_Tumor","CPTAC_Tumor"]}
        summary.append(dict(category=c,denominator=n,TCGA_count=cohort_counts["TCGA_Tumor"],
                            CPTAC_count=cohort_counts["CPTAC_Tumor"],TCGA_CPTAC_union_count=int(tumor[c]),
                            GTEx_count=int(normal[c]),TCGA_CPTAC_union_percent=100*tumor[c]/n,
                            GTEx_percent=100*normal[c]/n,fisher_two_sided_p=pv))
    pd.DataFrame(summary).to_csv(TABLE_DIR/"Figure3I_machinery_category_counts.tsv",sep="\t",index=False)
    membership = m.assign(rhythmic=pd.to_numeric(m.p,errors="coerce") < CONFIG["as_p"])
    membership.to_csv(TABLE_DIR/"Figure3I_machinery_gene_membership.tsv",sep="\t",index=False)
    ax.set_xticks(x); ax.set_xticklabels([str(c).replace(" ", "\n") for c in cats])
    _style(ax, None, "Rhythmic machinery genes (%)", f"Machinery rhythmicity (raw p < {CONFIG['as_p']:g})")
    ax.set_ylim(0,115)
    ax.legend(fontsize=FONT["legend"], frameon=False)
    return ax


def _marginal_phase(subfig, data, cohort):
    d = _data(data); as_df = _event_frame(d, cohort); ex = d['rhythmic_expression'][cohort]
    ax = _axes(subfig, polar=True)
    summaries = []; tables = []
    for frame, color, label in ((as_df, AS_COLOR, 'SE'), (ex, EXPR_COLOR, 'mRNA')):
        hours = _acrophase_hours(frame); theta = _theta(hours)
        grid, den = _kde(theta)
        ax.plot(grid, den, color=color, lw=1.1, label=f'{label} (n={len(frame)})')
        ax.fill_between(grid, 0, den, color=color, alpha=.12)
        r2, axis_h = _axial_orientation(theta); summaries.append((r2, axis_h))
        for h in [axis_h, (axis_h+12)%24]:
            ax.axvline(h*np.pi/12, color=color, lw=.55, ls='--')
        tables.append(pd.DataFrame({'phase_h':grid*12/np.pi,'density':den,'population':label}))
    delta_r2 = summaries[0][0]-summaries[1][0]
    delta_axis = ((summaries[0][1]-summaries[1][1]+6)%12)-6
    w,pval = _watson_wheeler(_acrophase_hours(as_df), _acrophase_hours(ex))
    _polar_clock(ax)
    ax.set_title(f'{_cohort_label(cohort)}\nΔ axis={delta_axis:.2f} h\nΔ R₂={delta_r2:+.2f}; WW p={pval:.2g}',
                 fontsize=FONT['title'], fontweight='bold', pad=5)
    ax.legend(fontsize=FONT['legend'], frameon=False, loc='upper center', bbox_to_anchor=(.5,-.13))
    pd.concat(tables,ignore_index=True).to_csv(TABLE_DIR/f'{cohort}_marginal_phase_density.tsv',sep='\t',index=False)
    pd.DataFrame([dict(cohort=cohort,delta_R2=delta_r2,delta_axis_h=delta_axis,watson_w=w,watson_p=pval)]).to_csv(
        TABLE_DIR/f'{cohort}_marginal_phase_comparison.tsv',sep='\t',index=False)
    return ax

def draw_3j(subfig, data=None): return _marginal_phase(subfig, data, "GTEx_Normal")
def draw_3k(subfig, data=None): return _marginal_phase(subfig, data, "TCGA_Tumor")


def draw_3l(subfig, data=None):
    a = _data(data)["amplitude_source"].copy(); ax = _axes(subfig)
    groups = ["no rhythmic SE", "with rhythmic SE"]
    a["group"] = np.where(a.has_rhythmic_se, groups[1], groups[0])
    cohorts = [c for c in ("GTEx_Normal", "TCGA_Tumor") if c in set(a.cohort)]
    positions=[]; values=[]; colors=[]; labels=[]
    for j,c in enumerate(cohorts):
        for i,g in enumerate(groups):
            vals=pd.to_numeric(a.loc[(a.cohort==c)&(a.group==g),"log_amplitude"],errors="coerce").dropna().to_numpy()
            positions.append(j*3+i); values.append(vals); colors.append(_cohort_colour(c) if i else "#BDBDBD"); labels.append(g.replace(" ","\n"))
    vp=ax.violinplot(values,positions=positions,widths=.8,showmeans=False,showmedians=False,showextrema=False)
    for body,color in zip(vp["bodies"],colors): body.set_facecolor(color);body.set_edgecolor(".25");body.set_alpha(.75)
    for pos,vals in zip(positions,values):
        q=np.quantile(vals,[.25,.5,.75]);ax.vlines(pos,q[0],q[2],color=".15",lw=1.2);ax.hlines(q[1],pos-.16,pos+.16,color=".15",lw=1.2)
    for j,c in enumerate(cohorts):
        aa,bb=values[2*j],values[2*j+1];pval=stats.mannwhitneyu(aa,bb,alternative="two-sided").pvalue;y=max(np.max(aa),np.max(bb));ax.text(j*3+.5,y+.08,f"MWU p={pval:.2g}",ha="center",fontsize=FONT["annotation"])
    ax.set_xticks(positions);ax.set_xticklabels(labels)
    _style(ax, None, "log₂(2 × amplitude + 0.01)", "Expression amplitude by rhythmic SE")
    for j,c in enumerate(cohorts): ax.text(j*3+.5,.01,_cohort_label(c),ha="center",transform=ax.get_xaxis_transform(),fontsize=FONT["annotation"])
    return ax


def _effects(subfig, data, contrasts):
    e = _data(data)["amplitude_effects"].copy()
    reference, target = contrasts
    selected = []
    for cohort in (reference, target):
        row = e.loc[e.cohort.eq(cohort) & e.effect.eq('with_rhythmic_SE_minus_no_rhythmic_SE')]
        if len(row) != 1: raise ValueError(f"Expected one amplitude effect for {cohort}")
        selected.append(row.iloc[0])
    contrast = e.loc[e.cohort.eq(target) & e.effect.eq(f'{target}_effect_minus_{reference}_effect')]
    if len(contrast) != 1: raise ValueError(f"Expected one amplitude contrast for {target}")
    selected.append(contrast.iloc[0]); e = pd.DataFrame(selected)
    ax = _axes(subfig)
    labels = [_cohort_label(reference).replace(' ', '\n'), _cohort_label(target).replace(' ', '\n'), 'Tumor −\nnormal']
    for i, (_, row) in enumerate(e.iterrows()):
        color = _cohort_colour(row.cohort) if i < 2 else '#777777'
        ax.errorbar(row.estimate, i, xerr=[[row.estimate-row.ci_low],[row.ci_high-row.estimate]],
                    fmt='o', color=color, capsize=2, ms=4)
    ax.axvline(0, color='.5', lw=.7, ls='--')
    ax.set_yticks(range(3), labels); ax.set_ylim(2.5,-.5)
    _style(ax, 'Median log₂ ratio\nwith SE / no SE', None, 'Amplitude effect\n(bootstrap 95% CI)')
    return ax

def draw_3m(subfig, data=None): return _effects(subfig, data, ("GTEx_Normal", "TCGA_Tumor"))


def _pvalue_stars(p):
    """Conventional raw-P ranges; exact boundaries belong to the larger range."""
    if not np.isfinite(p): return "NA"
    for threshold, label in [(1e-4, "****"), (1e-3, "***"), (1e-2, "**"), (.05, "*")]:
        if p < threshold: return label
    return "ns"


def _binding(subfig, data, scope, significance_stars=False):
    b=_data(data)["bindings"].copy(); b=b.loc[b.scope.eq(scope)]
    if b.empty: raise ValueError(f"No {scope} binding summary.")
    b=b.set_index("cohort").reindex(tuple(COHORT_INFO)).dropna(how="all").reset_index(); ax=_axes(subfig); x=np.arange(len(b))
    lower=(b.frac-b.ci_low).clip(lower=0)*100; upper=(b.ci_high-b.frac).clip(lower=0)*100
    ax.bar(x,b.frac*100,color=[_cohort_colour(c) for c in b.cohort], yerr=np.vstack([lower,upper]),capsize=2)
    for i,r in enumerate(b.itertuples()):
        ax.text(i,r.ci_high*100+1,f'{int(r.n_with)}/{int(r.n_total)}\n{r.frac*100:.1f}%',ha='center',fontsize=FONT['annotation'])
    reference = b.loc[b.cohort.eq('GTEx_Normal')].iloc[0]
    for i,r in enumerate(b.itertuples()):
        if r.cohort != 'GTEx_Normal':
            _, pval = stats.fisher_exact([[r.n_with,r.n_total-r.n_with],
                                          [reference.n_with,reference.n_total-reference.n_with]])
            annotation = _pvalue_stars(pval) if significance_stars else f'p={pval:.1g}'
            ax.text(i,90 if scope=='SE' else 93,annotation,ha='center',rotation=0,fontsize=FONT['annotation'])
    ax.set_ylim(0,103)
    ax.set_xticks(x); ax.set_xticklabels([_cohort_label(c).replace(" ","\n") for c in b.cohort], rotation=0, ha="center")
    _style(ax,None,"Events with clock-TF-bound host (%)",f"Clock-TF binding ({scope})\nEvent-weighted; p vs GTEx")
    return ax
def draw_3n(subfig,data=None): return _binding(subfig,data,"SE",significance_stars=True)
def draw_s9h(subfig,data=None): return _binding(subfig,data,"all")


def draw_s8b(subfig, data=None):
    m=_data(data)["machinery"].copy(); cohorts=[c for c in tuple(COHORT_INFO) if c in set(m.cohort)]
    # Fixed coordinates preserve the tall 81-row map without forcing long
    # category labels through constrained-layout calculations.
    ax1=subfig.add_axes([.18,.07,.30,.86]); ax2=subfig.add_axes([.65,.07,.28,.86],sharey=ax1)
    ax1.set_in_layout(False); ax2.set_in_layout(False)
    cats=list(_data(data).get("machinery_categories",{})) or list(pd.unique(m.category)); genes=[]; breaks=[]
    for cat in cats:
        group=sorted(m.loc[m.category.eq(cat),"gene_symbol"].dropna().unique()); breaks.append((len(genes),cat));genes.extend(group)
    pmat=np.full((len(genes),len(cohorts)),np.nan); ph=np.full_like(pmat,np.nan)
    for j,c in enumerate(cohorts):
        z=m.loc[m.cohort.eq(c)].set_index("gene_symbol")
        for i,g in enumerate(genes):
            if g in z.index:
                row=z.loc[g]; row=row.iloc[0] if isinstance(row,pd.DataFrame) else row
                pmat[i,j]=np.clip(-np.log10(max(row.p,1e-10))*np.sign(row.amplitude),-5,5)
                if row.p < CONFIG["as_p"]: ph[i,j]=_acrophase_hours(pd.DataFrame([row]))[0]
    im1=ax1.imshow(pmat,cmap="RdBu_r",vmin=-5,vmax=5,aspect="auto"); im2=ax2.imshow(ph,cmap="twilight",vmin=0,vmax=24,aspect="auto")
    for ax,title in ((ax1,"signed −log₁₀(p)"),(ax2,"phase (significant only)")):
        ax.set_xticks(range(len(cohorts))); ax.set_xticklabels([_cohort_label(c) for c in cohorts],rotation=25,ha="right"); ax.set_yticks(range(len(genes))); ax.set_yticklabels(genes, fontsize=FONT["tick"]); ax.set_title(title,fontsize=FONT["title"],fontweight="bold"); ax.tick_params(labelsize=FONT["tick"])
        for start,cat in breaks:
            if start: ax.axhline(start-.5,color="black",lw=.5)
            if ax is ax1:
                stop=next((b for b,_ in breaks if b>start),len(genes))
                ax.text(-.47,(start+stop-1)/2,str(cat),ha="center",va="center",rotation=90,
                        fontsize=FONT["annotation"],fontweight="bold",transform=ax.get_yaxis_transform())
    # White stars retain the raw-p < 0.05 call on the signed rhythmicity map.
    for i,g in enumerate(genes):
        for j,c in enumerate(cohorts):
            row=m.loc[(m.cohort==c)&(m.gene_symbol==g)]
            if len(row) and float(row.iloc[0].p)<CONFIG["as_p"]: ax1.text(j,i,"*",ha="center",va="center",fontsize=FONT["annotation"],color="white")
    ax2.tick_params(axis="y",left=False,labelleft=False)
    for left,im,label in ((.18,im1,"signed −log₁₀(p)"),(.65,im2,"acrophase (h)")):
        cax=subfig.add_axes([left,.974,.27,.009]);cax.set_in_layout(False)
        cb=subfig.colorbar(im,cax=cax,orientation="horizontal")
        cb.ax.tick_params(labelsize=FONT["tick"],pad=1)
        cb.ax.set_title(label,fontsize=FONT["axis_title"],fontweight="normal",pad=2)
    return ax1


def _metrics(data):
    met = _data(data)["circular_metrics"].copy()
    required = {"cohort", "AS_R2", "AS_ci_low", "AS_ci_high", "gene_R2", "gene_ci_low", "gene_ci_high", "delta_R2", "permutation_p", "axis_separation_h", "watson_p", "watson_w"}
    missing = required.difference(met.columns)
    if missing: raise ValueError(f"circular_metrics is missing {sorted(missing)}")
    return met.set_index("cohort").reindex(tuple(COHORT_INFO)).dropna(how="all").reset_index()


def _metric_xlabels(ax, met):
    x = np.arange(len(met)); ax.set_xticks(x)
    ax.set_xticklabels([_cohort_label(c).replace(" ","\n") for c in met.cohort], rotation=0, ha="center")
    return x


def draw_s9a(subfig, data=None):
    met = _metrics(data); ax = _axes(subfig); x = _metric_xlabels(ax, met); off = .12
    for feature, color, offset in (("AS", AS_COLOR, -off), ("gene", EXPR_COLOR, off)):
        v=met[f"{feature}_R2"]; lo=v-met[f"{feature}_ci_low"]; hi=met[f"{feature}_ci_high"]-v
        ax.errorbar(x+offset,v,yerr=np.vstack([lo.clip(lower=0),hi.clip(lower=0)]),fmt="o",ms=4,capsize=2,color=color,label="SE events" if feature=="AS" else "host genes")
    for i,r in met.iterrows(): ax.plot([x[i]-off,x[i]+off],[r.AS_R2,r.gene_R2],color=".75",lw=.7,zorder=0)
    ax.set_ylim(0,1); _style(ax,None,"Axial R₂","Axial polarization (95% bootstrap CI)"); ax.legend(fontsize=FONT["legend"],frameon=False)
    return ax


def draw_s9b(subfig, data=None):
    met=_metrics(data);ax=_axes(subfig);x=_metric_xlabels(ax,met); vals=met.delta_R2
    ax.bar(x,vals,color=np.where(vals>=0,AS_COLOR,EXPR_COLOR));ax.axhline(0,color=".3",lw=.7)
    limit=max(.05,float(np.nanmax(np.abs(vals)))*1.45);ax.set_ylim(-limit,limit)
    for i,r in met.iterrows(): ax.text(i,r.delta_R2+np.sign(r.delta_R2 or 1)*limit*.06,f"p={r.permutation_p:.3g}",ha="center",va="bottom" if r.delta_R2>=0 else "top",fontsize=FONT["annotation"])
    _style(ax,None,"Δ axial R₂ (SE − host)","Polarization difference\n(label permutation)");return ax


def draw_s9c(subfig, data=None):
    met=_metrics(data);ax=_axes(subfig);x=_metric_xlabels(ax,met);vals=met.axis_separation_h
    ax.bar(x,vals,color="#009E73");ax.set_ylim(0,6)
    for i,v in enumerate(vals):ax.text(i,v+.12,f"{v:.1f} h",ha="center",fontsize=FONT["annotation"])
    _style(ax,None,"Absolute axial separation (h)","Second-moment axis separation");return ax


def draw_s9d(subfig, data=None):
    met=_metrics(data);ax=_axes(subfig);x=_metric_xlabels(ax,met);score=-np.log10(met.watson_p.clip(lower=np.nextafter(0,1)))
    ax.bar(x,score,color="#CC79A7");ax.set_ylim(0,float(score.max())+.4);ax.axhline(-np.log10(.05),color=".3",lw=.7,ls="--")
    for i,r in met.iterrows():ax.text(i,score.iloc[i]+.06,f"W={r.watson_w:.2f}\np={r.watson_p:.3g}",ha="center",fontsize=FONT["annotation"])
    _style(ax,None,"−log₁₀(Watson–Wheeler p)","Watson–Wheeler two-sample test");return ax


def _normalized_accumulation(subfig, data, cohorts):
    """Legacy panel-B survival curves, with deterministic bootstrap bands."""
    a=_data(data)["amplitude_source"].copy();ax=_axes(subfig);rng=np.random.default_rng(CONFIG["seed"])
    export_rows=[]
    for cohort in cohorts:
        d=a.loc[a.cohort.eq(cohort)]
        for flag,style,label in ((False,"-","no rhythmic SE"),(True,"--","with rhythmic SE")):
            values=pd.to_numeric(d.loc[d.has_rhythmic_se.eq(flag),"normalized_log_amplitude"],errors="coerce").dropna().to_numpy()
            if len(values)<2: raise ValueError(f"{cohort} has insufficient {label} amplitudes.")
            grid=np.linspace(values.min(),values.max(),160);curve=100*(len(values)-np.searchsorted(np.sort(values),grid,side="left"))/len(values)
            boots=np.empty((CONFIG["curve_bootstrap_n"],len(grid)))
            for b in range(len(boots)):
                sample=np.sort(rng.choice(values,size=len(values),replace=True));boots[b]=100*(len(sample)-np.searchsorted(sample,grid,side="left"))/len(sample)
            lo,hi=np.quantile(boots,[.025,.975],axis=0);color=_cohort_colour(cohort)
            ax.plot(grid,curve,color=color,ls=style,lw=1.2,label=f"{_cohort_label(cohort)}; {label}")
            ax.fill_between(grid,lo,hi,color=color,alpha=.10)
            export_rows.append(pd.DataFrame({"cohort":cohort,"group":label,"threshold_normalized_log_amplitude":grid,"percent_at_or_above":curve,"ci_low":lo,"ci_high":hi,"n":len(values)}))
    table_dir = _data(data).get("TABLE_DIR", globals().get("TABLE_DIR"))
    if table_dir is not None:
        table_dir=Path(table_dir); table_dir.mkdir(parents=True, exist_ok=True)
        suffix="_".join(c.lower().replace("_","-") for c in cohorts)
        pd.concat(export_rows,ignore_index=True).to_csv(table_dir / f"normalized_amplitude_accumulation_{suffix}.csv",index=False)
    _style(ax,"Cohort-normalized log amplitude","Genes ≥ threshold (%)","Amplitude accumulation (bootstrap 95% CI)")
    ax.set_ylim(0,100);ax.legend(fontsize=FONT["legend"],frameon=False,loc="best");return ax
def draw_s9e(subfig,data=None):return _normalized_accumulation(subfig,data,("GTEx_Normal","TCGA_Tumor"))
def draw_s9f(subfig,data=None):return _normalized_accumulation(subfig,data,("GTEx_Normal","CPTAC_Tumor"))
def draw_s9g(subfig,data=None):return _effects(subfig,data,("GTEx_Normal","CPTAC_Tumor"))


def _phase_diff(subfig,data,cohort):
    d=_data(data)["matched"][cohort];ax=_axes(subfig,polar=True)
    # Legacy Cell 83 uses the full signed -12..12 h domain in 24 angular bins.
    hour_bins=np.linspace(-12,12,25);centers=(hour_bins[:-1]+hour_bins[1:])/2
    theta=centers*2*np.pi/24;width=2*np.pi/24
    for flag,color,label in ((True,PEAK_COLOR,"clock-TF peak"),(False,NO_PEAK_COLOR,"no peak")):
        vals=pd.to_numeric(d.loc[d.has_clock_peak.eq(flag),"phase_diff"],errors="coerce").dropna().clip(-12,12)
        counts,_=np.histogram(vals,bins=hour_bins);ax.bar(theta,counts,width=width,color=color,alpha=.52,label=f"{label} (n={len(vals)})")
    ax.set_theta_zero_location("E");ax.set_theta_direction(1)
    ticks=np.array([-12,-6,0,6]);ax.set_xticks(ticks*2*np.pi/24)
    ax.set_xticklabels(["±12 h","−6 h","0 h","+6 h"],fontsize=FONT["tick"])
    ax.set_thetalim(-np.pi,np.pi)
    ax.set_title(f"{_cohort_label(cohort)}\nAS − host phase (h)",fontsize=FONT["title"],fontweight="bold",pad=5);ax.legend(fontsize=FONT["legend"],frameon=False,loc="upper center",bbox_to_anchor=(.5,-.15))
    return ax
def draw_s9i(subfig,data=None):return _phase_diff(subfig,data,"CPTAC_Tumor")
def draw_s9j(subfig,data=None):return _phase_diff(subfig,data,"TCGA_Tumor")
def draw_s9k(subfig,data=None):return _phase_diff(subfig,data,"GTEx_Normal")


PANEL_SPECS = {
    "3B": (draw_3b,"se_phase_density_gtex",1,2), "3C": (draw_3c,"se_phase_density_tcga",1,2),
    "3D": (draw_3d,"slk_psi_phase",2,2), "3E": (draw_3e,"slk_junction_counts",2,2),
    "3F": (draw_3f,"se_heatmap_tcga",2,2), "3G": (draw_3g,"machinery_phase_tcga",1,2),
    "3H": (draw_3h,"machinery_phase_gtex",1,2), "3I": (draw_3i,"machinery_consistency",2,2),
    "3J": (draw_3j,"marginal_phase_gtex",1,2), "3K": (draw_3k,"marginal_phase_tcga",1,2),
    "3L": (draw_3l,"expression_amplitude_source",2,2), "3M": (draw_3m,"expression_amplitude_effect",1,2), "3N": (draw_3n,"clock_tf_binding_se",1,2),
    "S7B": (draw_s7b,"se_heatmap_cptac",2,2), "S7C": (draw_s7c,"se_heatmap_gtex",2,2),
    "S8A": (draw_s8a,"machinery_phase_cptac",1,2), "S8B": (draw_s8b,"machinery_rhythmicity_heatmaps",3,6),
    "S9A": (draw_s9a,"axial_r2_bootstrap",1,2), "S9B": (draw_s9b,"delta_axial_r2_permutation",1,2), "S9C": (draw_s9c,"second_moment_axis_separation",1,2), "S9D": (draw_s9d,"watson_wheeler_test",1,2),
    "S9E": (draw_s9e,"normalized_amplitude_gtex_tcga",2,2), "S9F": (draw_s9f,"normalized_amplitude_gtex_cptac",2,2), "S9G": (draw_s9g,"amplitude_effect_gtex_cptac",2,2), "S9H": (draw_s9h,"clock_tf_binding_all",2,2),
    "S9I": (draw_s9i,"phase_diff_clock_tf_cptac",1,2), "S9J": (draw_s9j,"phase_diff_clock_tf_tcga",1,2), "S9K": (draw_s9k,"phase_diff_clock_tf_gtex",1,2),
}
_PLOT_PANELS = {key: {"draw": fun, "content": content, "width": width, "height": height} for key,(fun,content,width,height) in PANEL_SPECS.items()}
try:
    PANELS.update(_PLOT_PANELS)
except NameError:
    PANELS = _PLOT_PANELS

# Figure 3 layout requested by the author: four rows, with a taller E/F row.
for key in ["3" + letter for letter in "ABCDEFGHIJKLMN"]:
    PANELS[key].update(width=1, height=1)
PANELS["3E"].update(width=3, height=1.5, content="slk_bam_sashimi")
PANELS["3F"].update(width=1, height=1.5)

# S9 uses square panels; the third row is evenly spaced during assembly.
for letter in "ABCDEFGHIJK":
    PANELS["S9" + letter].update(width=1, height=1)


## 5. Individual panels, assembled figures, and export validation

### Publication export helpers and physical layout

In [ ]:
# Cell 8: Publication export helpers and physical layout
import textwrap
from io import BytesIO

def wrap_panel_titles(subfig, width):
    max_chars = {1: 23, 2: 44, 3: 62, 4: 80}[width]
    for ax in subfig.axes:
        for location in ('left', 'center', 'right'):
            title = ax.get_title(loc=location)
            if title:
                wrapped = '\n'.join(textwrap.fill(line, width=max_chars, break_long_words=False)
                                    for line in title.splitlines())
                ax.set_title(wrapped, loc=location, fontsize=FONT['title'], fontweight='bold')

def format_main_panel(key, subfig):
    """Compact Figure 3 typography without altering the supplementary panels."""
    if not key.startswith('3') or key == '3E':
        return
    ax = subfig.axes[0]
    polar_keys = {'3B','3C','3G','3H','3J','3K'}
    if key in polar_keys:
        handles, labels = ax.get_legend_handles_labels()
        if ax.get_legend() is not None:
            ax.get_legend().remove()
        ax.set_title('')
        ax.tick_params(axis='x',pad=0)
        ax.set_position([.12,.24,.76,.53])
        ax.set_in_layout(False)
        if key in {'3B','3C'}:
            cohort = 'GTEx_Normal' if key == '3B' else 'TCGA_Tumor'
            frame = _event_frame(DATA, cohort)
            r2, _ = _axial_orientation(_theta(_acrophase_hours(frame)))
            title = f"{cohort.split('_')[0]} SE"
            subfig.text(.5,.92,f"n={len(frame)}; Axial R₂={r2:.2f}",ha='center',va='top',
                        fontsize=FONT['annotation'],fontweight='normal')
            ax.set_position([.12,.10,.76,.68])
        elif key in {'3G','3H'}:
            cohort = 'TCGA_Tumor' if key == '3G' else 'GTEx_Normal'
            count = ((DATA['machinery'].cohort == cohort) & (DATA['machinery'].p < CONFIG['as_p'])).sum()
            title = f"{cohort.split('_')[0]} - APA machinery"
            subfig.text(.5,.92,f"P < {CONFIG['as_p']:g}; n={count}",ha='center',va='top',
                        fontsize=FONT['annotation'],fontweight='normal')
            from matplotlib.ticker import MaxNLocator, FormatStrFormatter
            radial_max = ax.get_ylim()[1]
            ticks = MaxNLocator(nbins=3, steps=[1,2,5,10]).tick_values(0, radial_max)
            ax.set_yticks(ticks[(ticks > 0) & (ticks <= radial_max)])
            ax.yaxis.set_major_formatter(FormatStrFormatter('%g'))
            ax.set_rlabel_position(90)
            for text in ax.texts:
                offsets = {'NOVA1':(3,-8),'CELF2':(3,15),'SRSF12':(3,2)} if cohort == 'TCGA_Tumor' else {'CELF2':(3,-8),'SRSF12':(3,4)}
                if text.get_text() in offsets:
                    text.set_ha('left');text.set_position(offsets[text.get_text()])
            legend = subfig.legend(handles, ['Core','SR','hnRNP','Other'], loc='lower center',
                                  ncol=2, fontsize=FONT['legend'], frameon=False,
                                  bbox_to_anchor=(.5,.002), columnspacing=.7, handletextpad=.3,
                                  borderaxespad=0, labelspacing=.15)
            legend.set_in_layout(False)
        else:
            cohort = 'GTEx_Normal' if key == '3J' else 'TCGA_Tumor'
            ase = _event_frame(DATA,cohort); genes = DATA['rhythmic_expression'][cohort]
            ar, aa = _axial_orientation(_theta(_acrophase_hours(ase)))
            gr, ga = _axial_orientation(_theta(_acrophase_hours(genes)))
            delta = ((aa-ga+6)%12)-6
            _, pvalue = _watson_wheeler(_acrophase_hours(ase),_acrophase_hours(genes))
            title = f"{cohort.split('_')[0]}: SE vs mRNA"
            subfig.text(.5,.90,f"Δaxis {delta:+.2f} h; ΔR₂ {ar-gr:+.2f}\nWW P={pvalue:.2g}",
                        ha='center',va='top',fontsize=FONT['annotation'])
            ax.set_position([.14,.235,.72,.455])
            legend = subfig.legend(handles, [f'SE (n={len(ase)})',f'mRNA (n={len(genes)})'],
                                  loc='lower center',fontsize=FONT['legend'],frameon=False,
                                  bbox_to_anchor=(.5,.002),borderaxespad=0,labelspacing=.15)
            legend.set_in_layout(False)
        subfig.text(.5,.995,title,ha='center',va='top',fontsize=FONT['title'],fontweight='bold')
    elif key == '3D':
        ax.set_title('SLK skipped exon')
        handles, labels = ax.get_legend_handles_labels()
        ax.legend(handles,['Samples','Fit','Selected'],frameon=False,fontsize=FONT['legend'],
                  loc='lower left',handlelength=1,handletextpad=.3,borderaxespad=.1,labelspacing=.15)
        for text in ax.texts:
            if getattr(text,'xy',(0,0))[0] > 20:
                text.set_ha('right'); text.set_position((-2,3))
    elif key == '3F':
        ax.set_title(f"TCGA SE (n={len(DATA['heatmaps']['TCGA_Tumor']['matrix'])})")
    elif key == '3I':
        ax.set_ylabel('Rhythmic genes (%)',fontsize=FONT['axis_title'])
        ax.set_xticklabels(['Core','SR','hnRNP','Other'])
        ax.set_title('APA machinery rhythmicity')
        handles, labels = ax.get_legend_handles_labels()
        ax.legend(handles,['TCGA/CPTAC','GTEx'],ncol=1,loc='upper left',frameon=False,
                  fontsize=FONT['legend'],handlelength=1,columnspacing=.6,handletextpad=.3)
        ax.set_ylim(0,140);ax.set_yticks([0,50,100])
    elif key == '3L':
        ax.set_title('Expression amplitude')
        ax.set_ylabel('log₂ amplitude',fontsize=FONT['axis_title'])
        ax.set_xticklabels(['−','+','−','+'])
        ax.set_xlabel('Rhythmic SE (− / +)',fontsize=FONT['axis_title'])
        for text in ax.texts:
            if text.get_text() in {'GTEx normal','TCGA tumor'}:
                text.set_text(text.get_text().split()[0])
    elif key == '3M':
        ax.set_title('Amplitude effect\n95% bootstrap CI')
        ax.set_xlabel('Median log₂ ratio\nSE+ / SE−',fontsize=FONT['axis_title'])
    elif key == '3N':
        ax.set_title('SE host clock-TF\nbinding (event %)')
        ax.set_ylabel('Events (%)',fontsize=FONT['axis_title'])

def format_s9_panel(key, subfig):
    """Keep annotations readable when S9 panels occupy one square module."""
    if not key.startswith('S9'): return
    ax = subfig.axes[0]
    if key[-1] in 'ABCDH':
        ax.set_xticklabels(['CPTAC','TCGA','GTEx'])
    if key == 'S9A':
        ax.set_title('Axial polarization\n95% bootstrap CI')
        ax.legend(fontsize=FONT['legend'],frameon=False,loc='upper left',
                  handlelength=1,handletextpad=.3,borderaxespad=.2,labelspacing=.2)
    elif key == 'S9B':
        ax.set_title('Polarization difference\nPermutation test')
        ax.set_ylabel('Δ axial R₂ (SE − host)')
    elif key == 'S9C':
        ax.set_title('Axial separation')
        ax.set_ylabel('Axis separation (h)')
    elif key == 'S9D':
        ax.set_title('Watson–Wheeler test')
        ax.set_ylabel('−log₁₀(P)')
    elif key in {'S9E','S9F'}:
        target = 'TCGA' if key == 'S9E' else 'CPTAC'
        ax.set_title(f'GTEx / {target}\nAmplitude accumulation')
        ax.set_xlabel('Normalized log₂ amplitude')
        ax.set_ylabel('Genes ≥ threshold (%)')
        handles, _ = ax.get_legend_handles_labels()
        ax.legend(handles,['GTEx −','GTEx +',f'{target} −',f'{target} +'],
                  ncol=2,loc='upper right',fontsize=FONT['legend'],frameon=False,
                  handlelength=1.2,handletextpad=.3,columnspacing=.5,
                  borderaxespad=.2,labelspacing=.2)
    elif key == 'S9G':
        ax.set_title('Amplitude effect\n95% bootstrap CI')
        ax.set_yticklabels(['GTEx','CPTAC','Δ effect'])
        ax.set_xlabel('Median log₂ ratio\nSE+ / SE−')
    elif key == 'S9H':
        ax.set_title('Clock-TF binding\nAll AS events')
        ax.set_ylabel('Events (%)')
    elif key[-1] in 'IJK':
        cohort = {'I':'CPTAC','J':'TCGA','K':'GTEx'}[key[-1]]
        handles, labels = ax.get_legend_handles_labels()
        ax.get_legend().remove(); ax.set_title('')
        ax.set_position([.17,.21,.66,.54]); ax.set_in_layout(False)
        ax.tick_params(axis='x',pad=0)
        from matplotlib.ticker import MaxNLocator
        ax.yaxis.set_major_locator(MaxNLocator(nbins=3,integer=True))
        subfig.text(.5,.995,cohort,ha='center',va='top',fontsize=FONT['title'],fontweight='bold')
        subfig.text(.5,.92,'AS − host phase (h)',ha='center',va='top',fontsize=FONT['annotation'])
        legend = subfig.legend(handles,labels,loc='lower center',bbox_to_anchor=(.5,.002),
                              fontsize=FONT['legend'],frameon=False,borderaxespad=0,
                              handlelength=1,handletextpad=.3,labelspacing=.15)
        legend.set_in_layout(False)

def format_pvalue_text(fig):
    """Use lowercase italic p (raw) and q (FDR-adjusted) statistical symbols."""
    import re
    symbol = r'(?<![A-Za-z0-9_])[pPqQ](?![A-Za-z0-9_])'
    for text in fig.findobj(Text):
        parts = re.split(r'(\$[^$]*\$)', text.get_text())
        for i, part in enumerate(parts):
            if i % 2 == 0:
                parts[i] = re.sub(symbol,
                    lambda match: r'$\mathit{' + match[0].lower() + '}$', part)
            else:
                # Normalize statistical symbols already in MathText; repeated
                # export calls leave the formatted result unchanged.
                part = re.sub(r'\\(?:mathit|mathrm|mathbf)\{([pPqQ])\}',
                    lambda match: r'\mathit{' + match[1].lower() + '}', part)
                parts[i] = re.sub(symbol, lambda match: match[0].lower(), part)
        text.set_text(''.join(parts))

def figure_folder(number):
    return 'figure3' if number == '3' else 'supplementary' + number[1:]

def panel_tag(subfig, letter):
    # Point-sized labels are added only after saving the label-free PNG.
    return subfig.text(0.006, 0.997, letter, transform=subfig.transSubfigure,
                       fontsize=FONT['tag'], fontweight='bold', ha='left', va='top')

def validate_figure_text(fig, stem):
    fig.canvas.draw()
    texts = [artist for artist in fig.findobj(Text)
             if artist.get_visible() and artist.get_text().strip()]
    too_small = [(t.get_text()[:100], t.get_fontsize()) for t in texts if t.get_fontsize() < 7 - 1e-6]
    if too_small:
        raise AssertionError(f'Text below 7 pt in {stem}: {too_small[:5]}')
    return min((t.get_fontsize() for t in texts), default=FONT['base'])

def export_figure(fig, stem, width, height, tagged_subfigures):
    assert width in (1, 2, 3, 4) and height > 0 and np.isfinite(height)
    target = FIGURE_DIR / stem
    target.parent.mkdir(parents=True, exist_ok=True)
    for subfig in fig.subfigs:
        subfig.patch.set_visible(False)
    format_pvalue_text(fig)
    min_pt = validate_figure_text(fig, stem)
    if stem.startswith(('figure3/Figure3_assembled','supplementary9/FigureS9_assembled')):
        number = 'S9' if stem.startswith('supplementary9/') else '3'
        layout_rows = []
        for sf, letter in tagged_subfigures:
            spec = PANELS[number+letter]
            actual = np.array([sf.bbox.width, sf.bbox.height]) / fig.dpi * 25.4
            expected = PANEL_MM*np.array([spec['width'],spec['height']])
            assert np.allclose(actual,expected,atol=.05), (letter,actual,expected)
            layout_rows.append(dict(panel=letter,width_mm=actual[0],height_mm=actual[1],
                                    left_mm=sf.bbox.x0/fig.dpi*25.4,
                                    bottom_mm=sf.bbox.y0/fig.dpi*25.4))
        pd.DataFrame(layout_rows).to_csv(TABLE_DIR/f'Figure{number}_panel_layout.tsv',sep='\t',index=False)
    # Do not use bbox_inches='tight': nominal module dimensions must remain fixed.
    fig.savefig(target.with_suffix('.png'), dpi=300, facecolor='white')
    tags = [panel_tag(sf, letter) for sf, letter in tagged_subfigures]
    validate_figure_text(fig, stem)
    fig.savefig(target.with_suffix('.pdf'), facecolor='white')
    for tag in tags:
        tag.remove()
    EXPORT_MANIFEST.append(dict(stem=stem, width_modules=width, height_modules=height,
                                width_mm=PANEL_MM*width, height_mm=PANEL_MM*height,
                                minimum_text_pt=min_pt, png_has_panel_labels=False,
                                pdf_has_panel_labels=True,
                                png=str(target.with_suffix('.png').relative_to(FIGURE_DIR)),
                                pdf=str(target.with_suffix('.pdf').relative_to(FIGURE_DIR))))
    # Embed a screen-resolution preview while retaining the 300 dpi export.
    preview = BytesIO()
    fig.savefig(preview, format='png', dpi=115, facecolor='white')
    display(Image(data=preview.getvalue(), width=min(1100, round(PANEL_MM*width/25.4*115))))
    plt.close(fig)

def export_panel(key):
    p = PANELS[key]
    number, letter = key[:-1], key[-1]
    width, height = p['width'], p['height']
    fig = plt.figure(figsize=(PANEL_MM*width/25.4, PANEL_MM*height/25.4), layout='constrained')
    subfig = fig.subfigures(1, 1)
    p['draw'](subfig)
    format_main_panel(key, subfig)
    format_s9_panel(key, subfig)
    wrap_panel_titles(subfig, width)
    export_figure(fig, f'{figure_folder(number)}/Figure{key}_{p["content"]}', width, height, [(subfig, letter)])

def export_assembly(number, rows, content, spread_rows=()):
    # Rows contain panel keys. All components are redrawn from source data at
    # their native physical dimensions, retaining vector text and fixed fonts.
    widths = [sum(PANELS[key]['width'] for key in row) for row in rows]
    assert all(width <= 4 for width in widths), (number, widths)
    heights = [max(PANELS[key]['height'] for key in row) for row in rows]
    fig = plt.figure(figsize=(PANEL_MM*4/25.4, PANEL_MM*sum(heights)/25.4), layout='constrained')
    grid_columns = 24 if spread_rows else 4
    units_per_module = grid_columns // 4
    grid = fig.add_gridspec(len(rows), grid_columns, height_ratios=heights, wspace=0, hspace=0)
    tags = []
    row_start = 0
    for r, row in enumerate(rows):
        col = 0
        for j, key in enumerate(row):
            p = PANELS[key]
            assert p['height'] == heights[r], 'Panels in a row must share its module height.'
            span = p['width'] * units_per_module
            start = round((j+.5)*grid_columns/len(row)-span/2) if r in spread_rows else col
            sf = fig.add_subfigure(grid[r, start:start+span])
            p['draw'](sf)
            format_main_panel(key, sf)
            format_s9_panel(key, sf)
            wrap_panel_titles(sf, p['width'])
            tags.append((sf, key[-1]))
            col += span
        row_start += heights[r]
    export_figure(fig, f'{figure_folder(number)}/Figure{number}_assembled_{content}', 4, sum(heights), tags)


### Figure 3 — individual panels and assembled figure

Compact labels: Core = core spliceosome, SR = SR proteins, hnRNP = hnRNP proteins, Other = other splicing regulators. SE−/SE+ indicate absence/presence of rhythmic SE; log₂ amplitude is log₂(2 × expression amplitude + 0.01). The individual and assembled E panels show real BAM coverage with the first-to-third exon skipping connection above the coverage tracks. In I, TCGA/CPTAC means rhythmic in either tumor cohort (the gene union), compared with GTEx; * denotes two-sided Fisher exact *p* < 0.05 for Core. Other categories have no displayed significance annotation. G–I use the requested “APA machinery” display titles; the underlying gene categories remain Core spliceosome, SR proteins, hnRNP proteins, and other splicing regulators.

Panel N stars compare each tumor cohort with GTEx using the existing two-sided Fisher exact test: * 0.01 ≤ *p* < 0.05; ** 0.001 ≤ *p* < 0.01; *** 0.0001 ≤ *p* < 0.001; **** *p* < 0.0001; ns *p* ≥ 0.05.

Panel E: SLK skipped-exon event 150506, GRCh38 chr10. Filled red tracks show unsmoothed aligned-read depth with independent y-axis scales; junction arcs show measured spanning-read counts. The filters require primary, QC-pass, uniquely mapped (NH=1), properly paired, unclipped 48-nt reads. Duplicate flags are not excluded; overlapping mates contribute separately to read depth. All 12 measured junction counts match the processed rMATS table. Phase and PSI labels come from the existing CHIRAL/rMATS analysis. Introns are compressed fivefold; the two exon models show event inclusion/skipping paths, not full transcripts. The first code cell configures `SLK_BAM_DIR` (hypothetical default `/path/to/figure3E_bams`), the four `SLK_BAM_PATHS`, and `SLK_USE_SOURCE_CACHE`. Set the directory directly or via `CHRONO_SLK_BAM_DIR`. When permitted, the code validates and loads the portable `Figure3E_SLK_bam_*` source bundle if BAMs are unavailable.


In [ ]:
# Cell 9: Figure 3 — individual panels and assembled figure
DATA['slk_bam'] = build_slk_bam_sources(DATA['slk']['representatives'], DATA['slk']['junction_counts'])
display(DATA['slk_bam']['qc'])
display(DATA['slk_bam']['junctions'])
for letter in 'ABCDEFGHIJKLMN':
    export_panel('3' + letter)
export_assembly('3', [['3A', '3B', '3C', '3D'], ['3E', '3F'],
                      ['3G', '3H', '3I', '3J'], ['3K', '3L', '3M', '3N']],
                'rhythmic_alternative_splicing')


### Supplementary Figure S7 — pathways and SE landscapes

In [ ]:
# Cell 10: Supplementary Figure S7 — pathways and SE landscapes
for letter in 'ABC':
    export_panel('S7' + letter)
export_assembly('S7', [['S7A'], ['S7B', 'S7C']], 'pathways_and_se_landscapes')


### Supplementary Figure S8 — splicing machinery

In [ ]:
# Cell 11: Supplementary Figure S8 — splicing machinery
for letter in 'AB':
    export_panel('S8' + letter)
# Preserve the original polar-left/heatmap-right organization at native sizes.
def export_s8_assembly():
    a, b = PANELS['S8A'], PANELS['S8B']
    assert a['width'] + b['width'] == 4
    height = max(a['height'], b['height'])
    fig = plt.figure(figsize=(PANEL_MM*4/25.4, PANEL_MM*height/25.4), layout='constrained')
    grid = fig.add_gridspec(height, 4, wspace=0, hspace=0)
    sa = fig.add_subfigure(grid[:a['height'], :a['width']]); a['draw'](sa)
    sb = fig.add_subfigure(grid[:b['height'], a['width']:]); b['draw'](sb)
    wrap_panel_titles(sa, a['width']); wrap_panel_titles(sb, b['width'])
    export_figure(fig, 'supplementary8/FigureS8_assembled_splicing_machinery', 4, height, [(sa,'A'),(sb,'B')])
export_s8_assembly()


### Supplementary Figure S9 — expression coupling and clock-TF binding
S9 panels are 52.5 mm squares: A–D in row 1, E–H in row 2, and I–K centered at equal intervals across row 3. The assembly is 210 × 157.5 mm. E/F legend −/+ indicates absence/presence of rhythmic SE; G Δ effect is the CPTAC effect minus the GTEx effect. Raw p-values use lowercase italic *p*; FDR-adjusted values use lowercase italic *q*.


In [ ]:
# Cell 12: Supplementary Figure S9 — expression coupling and clock-TF binding
for letter in 'ABCDEFGHIJK':
    export_panel('S9' + letter)
export_assembly('S9', [['S9A','S9B','S9C','S9D'], ['S9E','S9F','S9G','S9H'],
                      ['S9I','S9J','S9K']], 'as_expression_clock_tf_coupling', spread_rows=(2,))


### Export validation and reproducibility record

In [ ]:
# Cell 13: Export validation and reproducibility record
manifest = pd.DataFrame(EXPORT_MANIFEST)
assert len(manifest) == 34, f'Expected 30 individual panels and 4 assemblies; got {len(manifest)}.'
assert not manifest['stem'].duplicated().any()
assert (manifest['minimum_text_pt'] >= 7).all()
assert (~manifest['png_has_panel_labels']).all() and manifest['pdf_has_panel_labels'].all()
assert all((FIGURE_DIR / p).is_file() for p in manifest['png'])
assert all((FIGURE_DIR / p).is_file() for p in manifest['pdf'])
manifest.to_csv(TABLE_DIR / 'publication_export_manifest.csv', index=False)
software = {p: importlib.metadata.version(p) for p in
            ['numpy','pandas','scipy','matplotlib','scikit-learn','ipykernel']}
session = dict(python=sys.version, platform=platform.platform(), software=software,
               configuration=CONFIG, fonts=FONT, panel_mm=PANEL_MM,
               elapsed_seconds=round(time.time()-START_TIME, 2))
(TABLE_DIR / 'publication_session_info.json').write_text(json.dumps(session, indent=2))
display(manifest[['stem','width_mm','height_mm','minimum_text_pt']])
print(f'Exported {len(manifest)} PNG/PDF pairs; every panel and assembly is displayed above.')
print('Source tables:', TABLE_DIR)
print('Figure outputs:', FIGURE_DIR)
